## FD_abs

In [ ]:
import sys
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/MCX_data'
sys.path.append(folder_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import sys
import pickle
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from sklearn.preprocessing import StandardScaler

### Read the saved CSV

In [ ]:
import pandas as pd
import glob
import numpy as np
import sys
! pip install pmcx
from google.colab import drive
drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/MCX_data'
sys.path.append(folder_path)
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import os

csv_path = "/content/drive/MyDrive/MCX_data/result_folder/training_fd_110MHz.csv"

df = pd.read_csv(csv_path)

distances = [10, 20, 30, 40]
metrics = ["uac", "udc", "phase_rad"]

sub = df[df["sds_key"].isin(distances)].copy()

wl_order = sorted(sub["wavelength_index"].unique())
ids = np.array(sorted(sub["simulation_id"].unique()))

feature_blocks = []
feature_names = []

for d in distances:
    for wl_idx in wl_order:
        tmp = (
            sub[(sub["sds_key"] == d) & (sub["wavelength_index"] == wl_idx)]
            .set_index("simulation_id")
            .loc[ids, metrics]
        )

        feature_blocks.append(tmp.to_numpy(dtype=np.float64))

        wl_label = f"wl{wl_idx + 1}"
        feature_names.extend([
            f"uac_d{d}_{wl_label}",
            f"udc_d{d}_{wl_label}",
            f"phase_d{d}_{wl_label}"
        ])

X = np.concatenate(feature_blocks, axis=1)

print(X.shape)
print(feature_names)

np.save("fd_features_Nx24.npy", X)
np.save("fd_features_ids.npy", ids)

out_df = pd.DataFrame(X, columns=feature_names)
training_set = out_df

(10000, 24)
['uac_d10_wl1', 'udc_d10_wl1', 'phase_d10_wl1', 'uac_d10_wl2', 'udc_d10_wl2', 'phase_d10_wl2', 'uac_d20_wl1', 'udc_d20_wl1', 'phase_d20_wl1', 'uac_d20_wl2', 'udc_d20_wl2', 'phase_d20_wl2', 'uac_d30_wl1', 'udc_d30_wl1', 'phase_d30_wl1', 'uac_d30_wl2', 'udc_d30_wl2', 'phase_d30_wl2', 'uac_d40_wl1', 'udc_d40_wl1', 'phase_d40_wl1', 'uac_d40_wl2', 'udc_d40_wl2', 'phase_d40_wl2']


In [ ]:
import pandas as pd
import numpy as np
import os

csv_path = "/content/drive/MyDrive/MCX_data/result_folder/testing_fd_110MHz.csv"

df = pd.read_csv(csv_path)

distances = [10, 20, 30, 40]
metrics = ["uac", "udc", "phase_rad"]

sub = df[df["sds_key"].isin(distances)].copy()

wl_order = sorted(sub["wavelength_index"].unique())
ids = np.array(sorted(sub["simulation_id"].unique()))

feature_blocks = []
feature_names = []

for d in distances:
    for wl_idx in wl_order:
        tmp = (
            sub[(sub["sds_key"] == d) & (sub["wavelength_index"] == wl_idx)]
            .set_index("simulation_id")
            .loc[ids, metrics]
        )

        feature_blocks.append(tmp.to_numpy(dtype=np.float64))

        wl_label = f"wl{wl_idx + 1}"
        feature_names.extend([
            f"uac_d{d}_{wl_label}",
            f"udc_d{d}_{wl_label}",
            f"phase_d{d}_{wl_label}"
        ])

X = np.concatenate(feature_blocks, axis=1)

print(X.shape)
print(feature_names)

np.save("fd_features_Nx24.npy", X)
np.save("fd_features_ids.npy", ids)

out_df = pd.DataFrame(X, columns=feature_names)
testing_set = out_df

(1000, 24)
['uac_d10_wl1', 'udc_d10_wl1', 'phase_d10_wl1', 'uac_d10_wl2', 'udc_d10_wl2', 'phase_d10_wl2', 'uac_d20_wl1', 'udc_d20_wl1', 'phase_d20_wl1', 'uac_d20_wl2', 'udc_d20_wl2', 'phase_d20_wl2', 'uac_d30_wl1', 'udc_d30_wl1', 'phase_d30_wl1', 'uac_d30_wl2', 'udc_d30_wl2', 'phase_d30_wl2', 'uac_d40_wl1', 'udc_d40_wl1', 'phase_d40_wl1', 'uac_d40_wl2', 'udc_d40_wl2', 'phase_d40_wl2']


### GT

In [ ]:
GT_folder_train = '/content/drive/MyDrive/MCX_data/stage2_csv/'
GT_folder_test = '/content/drive/MyDrive/MCX_data/test_csv/'

In [ ]:
csv_files_train = glob.glob(os.path.join(GT_folder_train, '*.csv'))
GT_all_train = pd.concat([pd.read_csv(f) for f in csv_files_train], ignore_index=True)
csv_files_test = glob.glob(os.path.join(GT_folder_test, '*.csv'))
GT_all_test = pd.concat([pd.read_csv(f) for f in csv_files_test], ignore_index=True)

In [ ]:
import numpy as np

sorted_ids = [i + 1 for i in range(10000)]

# ensure ID is int
GT_all_train["ID"] = GT_all_train["ID"].astype(int)

# filter + order by ID = 1..10000
GT_filtered = (GT_all_train[GT_all_train["ID"].isin(sorted_ids)]
               .copy()
               .set_index("ID")
               .loc[sorted_ids]
               .reset_index())

target_cols = ["HBO1","HHB1", "HBO2", "HHB2", "d1", "a1", "a2", "b1", "b2"]   # <-- adjust if needed

# (optional) verify all columns exist
missing = [c for c in target_cols if c not in GT_filtered.columns]
if missing:
    raise KeyError(f"Missing columns in GT_filtered: {missing}. Available: {list(GT_filtered.columns)}")

Y = GT_filtered[target_cols].to_numpy(dtype=np.float32)  # shape (N, 5)
Y_train = Y  # keep as (N,5) for multi-output regression

print("y_train shape:", Y_train.shape)
print("first row:", dict(zip(target_cols, Y_train[0])))

y_train shape: (10000, 9)
first row: {'HBO1': np.float32(10.618102), 'HHB1': np.float32(12.007143), 'HBO2': np.float32(46.95982), 'HHB2': np.float32(26.97317), 'd1': np.float32(12.0), 'a1': np.float32(1.8359671), 'a2': np.float32(1.2800592), 'b1': np.float32(2.1788228), 'b2': np.float32(2.103345)}


In [ ]:
import numpy as np

sorted_ids = [i + 1 for i in range(1000)]

# ensure ID is int
GT_all_test["ID"] = GT_all_test["ID"].astype(int)

# filter + order by ID = 1..10000
GT_filtered = (GT_all_test[GT_all_test["ID"].isin(sorted_ids)]
               .copy()
               .set_index("ID")
               .loc[sorted_ids]
               .reset_index())

target_cols = ["HBO1","HHB1", "HBO2", "HHB2", "d1", "a1", "a2", "b1", "b2"]   # <-- adjust if needed

# (optional) verify all columns exist
missing = [c for c in target_cols if c not in GT_filtered.columns]
if missing:
    raise KeyError(f"Missing columns in GT_filtered: {missing}. Available: {list(GT_filtered.columns)}")

Y = GT_filtered[target_cols].to_numpy(dtype=np.float32)  # shape (N, 5)
Y_test = Y  # keep as (N,5) for multi-output regression

print("y_test shape:", Y_test.shape)
print("first row:", dict(zip(target_cols, Y_test[0])))


y_test shape: (1000, 9)
first row: {'HBO1': np.float32(10.618102), 'HHB1': np.float32(12.007143), 'HBO2': np.float32(46.95982), 'HHB2': np.float32(26.97317), 'd1': np.float32(12.0), 'a1': np.float32(1.8359671), 'a2': np.float32(1.2800592), 'b1': np.float32(2.1788228), 'b2': np.float32(2.103345)}


In [ ]:
X_train = training_set
X_test = testing_set

In [ ]:
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

(10000, 24) (1000, 24) (10000, 9) (1000, 9)


In [ ]:
def fit_typewise_normalizer(X_train):
    """
    Fit mean/std using ONLY training data, separately for
    AC columns, DC columns, and PHASE columns.

    Returns:
        stats: dict containing means/stds and column indices
    """
    X_train = np.asarray(X_train, dtype=np.float64)

    n_features = X_train.shape[1]
    if n_features % 3 != 0:
        raise ValueError("Expected number of features to be divisible by 3.")

    ac_idx    = np.arange(0, n_features, 3)
    dc_idx    = np.arange(1, n_features, 3)
    phase_idx = np.arange(2, n_features, 3)

    ac_mean = X_train[:, ac_idx].mean()
    ac_std  = X_train[:, ac_idx].std()
    dc_mean = X_train[:, dc_idx].mean()
    dc_std  = X_train[:, dc_idx].std()
    ph_mean = X_train[:, phase_idx].mean()
    ph_std  = X_train[:, phase_idx].std()

    eps = 1e-8
    ac_std = max(ac_std, eps)
    dc_std = max(dc_std, eps)
    ph_std = max(ph_std, eps)

    stats = {
        "ac_idx": ac_idx,
        "dc_idx": dc_idx,
        "phase_idx": phase_idx,
        "ac_mean": ac_mean,
        "ac_std": ac_std,
        "dc_mean": dc_mean,
        "dc_std": dc_std,
        "ph_mean": ph_mean,
        "ph_std": ph_std,
    }
    return stats


def transform_typewise(X, stats):
    """
    Apply the train-fitted type-wise normalization to X.
    """
    X = np.asarray(X, dtype=np.float64).copy()

    X[:, stats["ac_idx"]]    = (X[:, stats["ac_idx"]]    - stats["ac_mean"]) / stats["ac_std"]
    X[:, stats["dc_idx"]]    = (X[:, stats["dc_idx"]]    - stats["dc_mean"]) / stats["dc_std"]
    X[:, stats["phase_idx"]] = (X[:, stats["phase_idx"]] - stats["ph_mean"]) / stats["ph_std"]

    return X

### K-D LUT  algorithm

In [ ]:
import numpy as np
from sklearn.neighbors import KDTree

# ============================================================
# 1) Training-only normalization
#    Assumption:
#    columns are ordered like
#    [AC, DC, PHASE, AC, DC, PHASE, ..., AC, DC, PHASE]
#    so for 96 columns, there are 32 triplets
# ============================================================

def fit_typewise_normalizer(X_train):
    """
    Fit mean/std using ONLY training data, separately for
    AC columns, DC columns, and PHASE columns.

    Returns:
        stats: dict containing means/stds and column indices
    """
    X_train = np.asarray(X_train, dtype=np.float64)

    n_features = X_train.shape[1]
    if n_features % 3 != 0:
        raise ValueError("Expected number of features to be divisible by 3.")

    ac_idx    = np.arange(0, n_features, 3)
    dc_idx    = np.arange(1, n_features, 3)
    phase_idx = np.arange(2, n_features, 3)

    ac_mean = X_train[:, ac_idx].mean()
    ac_std  = X_train[:, ac_idx].std()
    dc_mean = X_train[:, dc_idx].mean()
    dc_std  = X_train[:, dc_idx].std()
    ph_mean = X_train[:, phase_idx].mean()
    ph_std  = X_train[:, phase_idx].std()

    eps = 1e-8
    ac_std = max(ac_std, eps)
    dc_std = max(dc_std, eps)
    ph_std = max(ph_std, eps)

    stats = {
        "ac_idx": ac_idx,
        "dc_idx": dc_idx,
        "phase_idx": phase_idx,
        "ac_mean": ac_mean,
        "ac_std": ac_std,
        "dc_mean": dc_mean,
        "dc_std": dc_std,
        "ph_mean": ph_mean,
        "ph_std": ph_std,
    }
    return stats


def transform_typewise(X, stats):
    """
    Apply the train-fitted type-wise normalization to X.
    """
    X = np.asarray(X, dtype=np.float64).copy()

    X[:, stats["ac_idx"]]    = (X[:, stats["ac_idx"]]    - stats["ac_mean"]) / stats["ac_std"]
    X[:, stats["dc_idx"]]    = (X[:, stats["dc_idx"]]    - stats["dc_mean"]) / stats["dc_std"]
    X[:, stats["phase_idx"]] = (X[:, stats["phase_idx"]] - stats["ph_mean"]) / stats["ph_std"]

    return X


# ============================================================
# 2) KNN/KDTree lookup + inverse-distance weighted prediction
# ============================================================

def kd_lookup_predict(X_train, Y_train, X_test, k=5, eps=1e-8):
    """
    Build KDTree on normalized training X and predict Y for X_test
    by inverse-distance weighted average of top-k neighbors.

    Returns:
        Y_pred         : (n_test, n_targets)
        neighbor_idx   : (n_test, k)
        neighbor_dist  : (n_test, k)
        norm_stats     : fitted normalization stats
        tree           : KDTree object
    """
    X_train = np.asarray(X_train, dtype=np.float64)
    Y_train = np.asarray(Y_train, dtype=np.float64)
    X_test  = np.asarray(X_test, dtype=np.float64)

    # fit normalization ONLY on training
    norm_stats = fit_typewise_normalizer(X_train)

    # transform train and test using train stats
    X_train_norm = transform_typewise(X_train, norm_stats)
    X_test_norm  = transform_typewise(X_test, norm_stats)

    # build KDTree
    tree = KDTree(X_train_norm, leaf_size=40, metric="euclidean")

    # query top-k nearest neighbors
    neighbor_dist, neighbor_idx = tree.query(X_test_norm, k=k)

    # gather neighbor targets
    # shape: (n_test, k, n_targets)
    Y_neighbors = Y_train[neighbor_idx]

    # inverse-distance weights
    weights = 1.0 / (neighbor_dist + eps)

    # handle exact matches safely
    zero_mask = neighbor_dist < eps
    has_zero = zero_mask.any(axis=1)

    if np.any(has_zero):
        weights[has_zero] = zero_mask[has_zero].astype(np.float64)

    # normalize weights row-wise
    weights = weights / weights.sum(axis=1, keepdims=True)

    # weighted average
    Y_pred = np.sum(Y_neighbors * weights[:, :, None], axis=1)

    return Y_pred, neighbor_idx, neighbor_dist, norm_stats, tree


# ============================================================
# 3) Evaluation with mean ± std, including MAPE
# ============================================================

def print_metrics_with_std(Y_true, Y_pred, title="kd_lookup", mape_eps=1e-8):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    if Y_true.ndim == 1:
        Y_true = Y_true.reshape(-1, 1)
    if Y_pred.ndim == 1:
        Y_pred = Y_pred.reshape(-1, 1)

    err = Y_pred - Y_true
    abs_err = np.abs(err)

    # safe MAPE denominator
    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0

    # overall
    overall_mae = abs_err.mean()
    overall_mae_std = abs_err.std()

    overall_rmse = np.sqrt(np.mean(err ** 2))
    per_sample_rmse = np.sqrt(np.mean(err ** 2, axis=1))
    overall_rmse_std = per_sample_rmse.std()

    overall_mape = ape.mean()
    overall_mape_std = ape.std()

    print(title)
    print(f"Overall MAE : {overall_mae:.6f} ± {overall_mae_std:.6f}")
    print(f"Overall RMSE: {overall_rmse:.6f} ± {overall_rmse_std:.6f}")
    print(f"Overall MAPE: {overall_mape:.6f} ± {overall_mape_std:.6f}")

    # per target
    for j in range(Y_true.shape[1]):
        err_j = err[:, j]
        abs_err_j = abs_err[:, j]
        ape_j = ape[:, j]

        mae_j = abs_err_j.mean()
        mae_j_std = abs_err_j.std()

        rmse_j = np.sqrt(np.mean(err_j ** 2))
        rmse_j_std = np.sqrt(err_j ** 2).std()

        mape_j = ape_j.mean()
        mape_j_std = ape_j.std()

        print(
            f"Target {j}: "
            f"MAE={mae_j:.6f} ± {mae_j_std:.6f}, "
            f"RMSE={rmse_j:.6f} ± {rmse_j_std:.6f}, "
            f"MAPE={mape_j:.6f} ± {mape_j_std:.6f}"
        )


# ============================================================
# 4) Run prediction
# ============================================================

k = 5

Y_pred_test, nn_idx, nn_dist, norm_stats, kd_tree = kd_lookup_predict(
    X_train=X_train,
    Y_train=Y_train,
    X_test=X_test,
    k=k
)

print("Y_pred_test shape:", Y_pred_test.shape)
print("Neighbor index shape:", nn_idx.shape)
print("Neighbor dist shape:", nn_dist.shape)


# ============================================================
# 5) Print evaluation
# ============================================================

print_metrics_with_std(Y_test, Y_pred_test, title="kd_lookup", mape_eps=1e-8)

Y_pred_test shape: (1000, 9)
Neighbor index shape: (1000, 5)
Neighbor dist shape: (1000, 5)
kd_lookup
Overall MAE : 1.124495 ± 2.105229
Overall RMSE: 2.386729 ± 1.475218
Overall MAPE: 14.028049 ± 29.376969
Target 0: MAE=0.759584 ± 0.857083, RMSE=1.145233 ± 0.857083, MAPE=7.111629 ± 9.701528
Target 1: MAE=0.703475 ± 0.761188, RMSE=1.036477 ± 0.761188, MAPE=11.799984 ± 17.079021
Target 2: MAE=3.977534 ± 4.024353, RMSE=5.658286 ± 4.024353, MAPE=10.603382 ± 11.753753
Target 3: MAE=2.733124 ± 2.672999, RMSE=3.822943 ± 2.672999, MAPE=11.888727 ± 13.085427
Target 4: MAE=0.811171 ± 0.901110, RMSE=1.212434 ± 0.901110, MAPE=5.666869 ± 6.910744
Target 5: MAE=0.227847 ± 0.288188, RMSE=0.367378 ± 0.288188, MAPE=5.612453 ± 6.199275
Target 6: MAE=0.405568 ± 0.407200, RMSE=0.574715 ± 0.407200, MAPE=18.366264 ± 23.510719
Target 7: MAE=0.113250 ± 0.119408, RMSE=0.164572 ± 0.119408, MAPE=19.307563 ± 45.388477
Target 8: MAE=0.388904 ± 0.371177, RMSE=0.537605 ± 0.371177, MAPE=35.895569 ± 60.316566


### Different KNN methods

In [ ]:
import numpy as np
from sklearn.neighbors import (
    KDTree,
    NearestNeighbors,
    KNeighborsRegressor,
    RadiusNeighborsRegressor,
)

# ============================================================
# 1) Training-only normalization
#    Assumption:
#    columns are ordered like
#    [AC, DC, PHASE, AC, DC, PHASE, ..., AC, DC, PHASE]
# ============================================================

def fit_typewise_normalizer(X_train):
    """
    Fit mean/std using ONLY training data, separately for
    AC columns, DC columns, and PHASE columns.
    """
    X_train = np.asarray(X_train, dtype=np.float64)

    n_features = X_train.shape[1]
    if n_features % 3 != 0:
        raise ValueError("Expected number of features to be divisible by 3.")

    ac_idx    = np.arange(0, n_features, 3)
    dc_idx    = np.arange(1, n_features, 3)
    phase_idx = np.arange(2, n_features, 3)

    ac_mean = X_train[:, ac_idx].mean()
    ac_std  = X_train[:, ac_idx].std()
    dc_mean = X_train[:, dc_idx].mean()
    dc_std  = X_train[:, dc_idx].std()
    ph_mean = X_train[:, phase_idx].mean()
    ph_std  = X_train[:, phase_idx].std()

    eps = 1e-8
    ac_std = max(ac_std, eps)
    dc_std = max(dc_std, eps)
    ph_std = max(ph_std, eps)

    stats = {
        "ac_idx": ac_idx,
        "dc_idx": dc_idx,
        "phase_idx": phase_idx,
        "ac_mean": ac_mean,
        "ac_std": ac_std,
        "dc_mean": dc_mean,
        "dc_std": dc_std,
        "ph_mean": ph_mean,
        "ph_std": ph_std,
    }
    return stats


def transform_typewise(X, stats):
    """
    Apply the train-fitted type-wise normalization to X.
    """
    X = np.asarray(X, dtype=np.float64).copy()

    X[:, stats["ac_idx"]]    = (X[:, stats["ac_idx"]]    - stats["ac_mean"]) / stats["ac_std"]
    X[:, stats["dc_idx"]]    = (X[:, stats["dc_idx"]]    - stats["dc_mean"]) / stats["dc_std"]
    X[:, stats["phase_idx"]] = (X[:, stats["phase_idx"]] - stats["ph_mean"]) / stats["ph_std"]

    return X


# ============================================================
# 2) Helper utilities
# ============================================================

def evaluate_predictions(Y_true, Y_pred, mape_eps=1e-8):
    """
    Return overall and per-target metrics with mean ± std.

    Definitions:
      - Overall MAE mean/std:
          computed from all absolute errors across all samples and targets
      - Overall RMSE mean/std:
          computed from sample-wise RMSE across targets
      - Overall MAPE mean/std:
          computed from all absolute percentage errors across all samples and targets
          using denominator max(|y_true|, mape_eps)
      - Per-target MAE mean/std:
          computed from absolute errors across samples for that target
      - Per-target RMSE mean/std:
          computed from per-sample root squared error for that target
          (for one target this is just absolute error, but we keep the label
           as RMSE for consistency with the original output style)
      - Per-target MAPE mean/std:
          computed from absolute percentage errors across samples for that target
          using denominator max(|y_true|, mape_eps)
    """
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    err = Y_pred - Y_true                     # (N, T)
    abs_err = np.abs(err)                    # (N, T)
    sq_err = err ** 2                        # (N, T)

    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0          # (N, T)

    # --------------------------------------------------------
    # Overall MAE = classic global MAE
    # Std = std of absolute errors over all entries
    # --------------------------------------------------------
    overall_mae = abs_err.mean()
    overall_mae_std = abs_err.std(ddof=0)

    # --------------------------------------------------------
    # Overall RMSE = classic global RMSE
    # Std = std of sample-wise RMSE across targets
    # --------------------------------------------------------
    overall_rmse = np.sqrt(sq_err.mean())
    sample_rmse = np.sqrt(sq_err.mean(axis=1))   # (N,)
    overall_rmse_std = sample_rmse.std(ddof=0)

    # --------------------------------------------------------
    # Overall MAPE = classic global MAPE with safe denominator
    # Std = std of absolute percentage errors over all entries
    # --------------------------------------------------------
    overall_mape = ape.mean()
    overall_mape_std = ape.std(ddof=0)

    per_target = []
    for j in range(Y_true.shape[1]):
        abs_j = abs_err[:, j]
        sq_j = sq_err[:, j]
        ape_j = ape[:, j]

        mae_j = abs_j.mean()
        mae_j_std = abs_j.std(ddof=0)

        rmse_j = np.sqrt(sq_j.mean())
        rmse_j_sample = np.sqrt(sq_j)            # = abs error for one target
        rmse_j_std = rmse_j_sample.std(ddof=0)

        mape_j = ape_j.mean()
        mape_j_std = ape_j.std(ddof=0)

        per_target.append({
            "target": j,
            "mae": mae_j,
            "mae_std": mae_j_std,
            "rmse": rmse_j,
            "rmse_std": rmse_j_std,
            "mape": mape_j,
            "mape_std": mape_j_std,
        })

    return {
        "overall_mae": overall_mae,
        "overall_mae_std": overall_mae_std,
        "overall_rmse": overall_rmse,
        "overall_rmse_std": overall_rmse_std,
        "overall_mape": overall_mape,
        "overall_mape_std": overall_mape_std,
        "per_target": per_target,
    }


def print_metrics(name, metrics):
    print(f"\n{name}")
    print(f"Overall MAE : {metrics['overall_mae']:.6f} ± {metrics['overall_mae_std']:.6f}")
    print(f"Overall RMSE: {metrics['overall_rmse']:.6f} ± {metrics['overall_rmse_std']:.6f}")
    print(f"Overall MAPE: {metrics['overall_mape']:.6f} ± {metrics['overall_mape_std']:.6f}")

    for row in metrics["per_target"]:
        print(
            f"Target {row['target']}: "
            f"MAE={row['mae']:.6f} ± {row['mae_std']:.6f}, "
            f"RMSE={row['rmse']:.6f} ± {row['rmse_std']:.6f}, "
            f"MAPE={row['mape']:.6f} ± {row['mape_std']:.6f}"
        )


def print_sorted_summary(results):
    """
    Print a compact summary sorted by RMSE.
    """
    print("\n" + "=" * 120)
    print("Sorted summary (best overall RMSE first)")
    print("=" * 120)

    rows = []
    for name, obj in results.items():
        m = obj["metrics"]
        rows.append((
            name,
            m["overall_mae"],
            m["overall_mae_std"],
            m["overall_rmse"],
            m["overall_rmse_std"],
            m["overall_mape"],
            m["overall_mape_std"],
        ))

    rows.sort(key=lambda x: x[3])

    for name, mae_, mae_std_, rmse_, rmse_std_, mape_, mape_std_ in rows:
        print(
            f"{name:30s}  "
            f"MAE={mae_:10.6f} ± {mae_std_:10.6f}  "
            f"RMSE={rmse_:10.6f} ± {rmse_std_:10.6f}  "
            f"MAPE={mape_:10.6f} ± {mape_std_:10.6f}"
        )


def _safe_raw_distance_weights(distances, power=1.0, eps=1e-8):
    """
    Raw inverse-distance style weights.
    Shape in -> shape out.
    """
    distances = np.asarray(distances, dtype=np.float64)
    return 1.0 / np.power(np.maximum(distances, eps), power)


def make_inverse_square_weight_fn(eps=1e-8):
    def weight_fn(distances):
        return _safe_raw_distance_weights(distances, power=2.0, eps=eps)
    return weight_fn


def make_gaussian_weight_fn(sigma, eps=1e-8):
    sigma = float(max(sigma, eps))

    def weight_fn(distances):
        distances = np.asarray(distances, dtype=np.float64)
        return np.exp(-0.5 * (distances / sigma) ** 2)

    return weight_fn


def _normalize_manual_weights(distances, raw_weights, eps=1e-8):
    """
    Normalize weights row-wise, with exact-match handling.
    Works for:
      distances: (k,) or (n, k)
      raw_weights: same shape
    """
    distances = np.asarray(distances, dtype=np.float64)
    raw_weights = np.asarray(raw_weights, dtype=np.float64)

    squeeze_back = False
    if distances.ndim == 1:
        distances = distances[None, :]
        raw_weights = raw_weights[None, :]
        squeeze_back = True

    zero_mask = distances < eps
    has_zero = zero_mask.any(axis=1)

    if np.any(has_zero):
        raw_weights[has_zero] = zero_mask[has_zero].astype(np.float64)

    row_sum = raw_weights.sum(axis=1, keepdims=True)
    bad = row_sum.squeeze(-1) <= 0

    if np.any(bad):
        raw_weights[bad] = 1.0
        row_sum = raw_weights.sum(axis=1, keepdims=True)

    weights = raw_weights / row_sum

    if squeeze_back:
        return weights[0]
    return weights


def estimate_neighbor_scale(X_train_norm, k=5):
    """
    Estimate a robust scale from train-set kNN distances.
    Used for Gaussian weights and radius initialization.
    """
    X_train_norm = np.asarray(X_train_norm, dtype=np.float64)
    n_train = len(X_train_norm)
    k_eff = min(k + 1, n_train)

    nbrs = NearestNeighbors(
        n_neighbors=k_eff,
        algorithm="ball_tree",
        metric="minkowski",
        p=2,
        n_jobs=-1,
    )
    nbrs.fit(X_train_norm)

    dists, _ = nbrs.kneighbors(X_train_norm)

    # Exclude self-distance at column 0 when possible
    if dists.shape[1] >= 2:
        useful = dists[:, 1:]
    else:
        useful = dists

    positive = useful[useful > 0]
    if positive.size == 0:
        return 1.0, 1.0

    sigma = float(np.median(positive))
    radius = float(np.quantile(useful[:, -1], 0.95))
    sigma = max(sigma, 1e-8)
    radius = max(radius, 1e-8)
    return sigma, radius


# ============================================================
# 3) Exact custom KDTree Shepard-style baseline
#    This reproduces your original logic directly.
# ============================================================

def kd_lookup_predict_exact(X_train_norm, Y_train, X_test_norm, k=5, eps=1e-8):
    """
    Exact KDTree lookup + inverse-distance weighted average.
    Returns:
        Y_pred, neighbor_idx, neighbor_dist, tree
    """
    X_train_norm = np.asarray(X_train_norm, dtype=np.float64)
    Y_train = np.asarray(Y_train, dtype=np.float64)
    X_test_norm = np.asarray(X_test_norm, dtype=np.float64)

    tree = KDTree(X_train_norm, leaf_size=40, metric="euclidean")
    neighbor_dist, neighbor_idx = tree.query(X_test_norm, k=k)

    Y_neighbors = Y_train[neighbor_idx]
    raw_weights = _safe_raw_distance_weights(neighbor_dist, power=1.0, eps=eps)
    weights = _normalize_manual_weights(neighbor_dist, raw_weights, eps=eps)

    Y_pred = np.sum(Y_neighbors * weights[:, :, None], axis=1)
    return Y_pred, neighbor_idx, neighbor_dist, tree


# ============================================================
# 4) Safe RadiusNeighborsRegressor prediction
#    Uses official radius-neighbor search, with fallback to a
#    standard distance-weighted KNN if a query has no neighbors.
# ============================================================

def predict_radius_regressor_safe(model, X_test_norm, Y_train, fallback_model, eps=1e-8):
    """
    Manual prediction on top of RadiusNeighborsRegressor's official
    neighbor search, so rows with zero neighbors can fall back safely.
    """
    dists_list, idx_list = model.radius_neighbors(
        X_test_norm,
        return_distance=True,
        sort_results=True,
    )

    Y_train = np.asarray(Y_train, dtype=np.float64)
    fallback_pred = fallback_model.predict(X_test_norm)

    n_test = len(X_test_norm)
    n_targets = Y_train.shape[1]
    Y_pred = np.zeros((n_test, n_targets), dtype=np.float64)

    for i, (d, idx) in enumerate(zip(dists_list, idx_list)):
        if len(idx) == 0:
            Y_pred[i] = fallback_pred[i]
            continue

        y_neighbors = Y_train[idx]

        if model.weights in (None, "uniform"):
            raw_w = np.ones(len(idx), dtype=np.float64)
        elif model.weights == "distance":
            raw_w = _safe_raw_distance_weights(d, power=1.0, eps=eps)
        elif callable(model.weights):
            raw_w = np.asarray(model.weights(d), dtype=np.float64)
        else:
            raise ValueError(f"Unsupported radius weight mode: {model.weights}")

        w = _normalize_manual_weights(d, raw_w, eps=eps)
        Y_pred[i] = np.sum(y_neighbors * w[:, None], axis=0)

    return Y_pred


# ============================================================
# 5) Full benchmark runner
# ============================================================

def run_knn_benchmark(X_train, Y_train, X_test, Y_test, k=5, eps=1e-8, mape_eps=1e-8):
    """
    Benchmarks a set of official sklearn KNN regressors plus the
    exact custom KDTree baseline.

    Returns:
        results      : dict with predictions, metrics, and fitted models
        norm_stats   : fitted X normalizer
        X_train_norm : normalized train X
        X_test_norm  : normalized test X
    """
    X_train = np.asarray(X_train, dtype=np.float64)
    Y_train = np.asarray(Y_train, dtype=np.float64)
    X_test  = np.asarray(X_test, dtype=np.float64)
    Y_test  = np.asarray(Y_test, dtype=np.float64)

    k = int(min(k, len(X_train)))
    if k < 1:
        raise ValueError("k must be >= 1.")

    # -------------------------
    # Fit normalization ONLY on train
    # -------------------------
    norm_stats = fit_typewise_normalizer(X_train)
    X_train_norm = transform_typewise(X_train, norm_stats)
    X_test_norm  = transform_typewise(X_test, norm_stats)

    # -------------------------
    # Estimate scales from training neighborhood geometry
    # -------------------------
    sigma, radius = estimate_neighbor_scale(X_train_norm, k=k)

    print("Normalization fitted on training only.")
    print("X_train_norm shape:", X_train_norm.shape)
    print("X_test_norm shape :", X_test_norm.shape)
    print(f"Using k={k}")
    print(f"Estimated Gaussian sigma: {sigma:.6f}")
    print(f"Estimated radius         : {radius:.6f}")

    gaussian_weight_fn = make_gaussian_weight_fn(sigma=sigma, eps=eps)
    inverse_square_weight_fn = make_inverse_square_weight_fn(eps=eps)

    results = {}

    # ========================================================
    # A) Official KNeighborsRegressor implementations
    # ========================================================

    official_models = {
    "knn_uniform_l2": KNeighborsRegressor(
        n_neighbors=k,
        weights="uniform",
        algorithm="ball_tree",
        metric="minkowski",
        p=2,
        leaf_size=40,
        n_jobs=-1,
    ),
    "knn_distance_l2": KNeighborsRegressor(
        n_neighbors=k,
        weights="distance",
        algorithm="ball_tree",
        metric="minkowski",
        p=2,
        leaf_size=40,
        n_jobs=-1,
    ),
    "knn_uniform_l1": KNeighborsRegressor(
        n_neighbors=k,
        weights="uniform",
        algorithm="ball_tree",
        metric="minkowski",
        p=1,
        leaf_size=40,
        n_jobs=-1,
    ),
    "knn_distance_l1": KNeighborsRegressor(
        n_neighbors=k,
        weights="distance",
        algorithm="ball_tree",
        metric="minkowski",
        p=1,
        leaf_size=40,
        n_jobs=-1,
    ),
    "knn_inverse_square_l2": KNeighborsRegressor(
        n_neighbors=k,
        weights=inverse_square_weight_fn,
        algorithm="ball_tree",
        metric="minkowski",
        p=2,
        leaf_size=40,
        n_jobs=-1,
    ),
    "knn_inverse_square_l1": KNeighborsRegressor(
        n_neighbors=k,
        weights=inverse_square_weight_fn,
        algorithm="ball_tree",
        metric="minkowski",
        p=1,
        leaf_size=40,
        n_jobs=-1,
    ),
    "knn_gaussian_l2": KNeighborsRegressor(
        n_neighbors=k,
        weights=gaussian_weight_fn,
        algorithm="ball_tree",
        metric="minkowski",
        p=2,
        leaf_size=40,
        n_jobs=-1,
    ),
}

    for name, model in official_models.items():
        model.fit(X_train_norm, Y_train)
        Y_pred = model.predict(X_test_norm)
        metrics = evaluate_predictions(Y_test, Y_pred, mape_eps=mape_eps)

        results[name] = {
            "model": model,
            "Y_pred": Y_pred,
            "metrics": metrics,
        }

    # ========================================================
    # B) Official RadiusNeighborsRegressor implementations
    #    Safe fallback: if a test point has zero radius-neighbors,
    #    use standard distance-weighted kNN.
    # ========================================================

    fallback_knn = KNeighborsRegressor(
        n_neighbors=k,
        weights="distance",
        algorithm="ball_tree",
        metric="minkowski",
        p=2,
        leaf_size=40,
        n_jobs=-1,
    )
    fallback_knn.fit(X_train_norm, Y_train)

    radius_models = {
        "radius_uniform_l2": RadiusNeighborsRegressor(
            radius=radius,
            weights="uniform",
            algorithm="ball_tree",
            metric="minkowski",
            p=2,
            leaf_size=40,
            n_jobs=-1,
        ),
        "radius_distance_l2": RadiusNeighborsRegressor(
            radius=radius,
            weights="distance",
            algorithm="ball_tree",
            metric="minkowski",
            p=2,
            leaf_size=40,
            n_jobs=-1,
        ),
        "radius_gaussian_l2": RadiusNeighborsRegressor(
            radius=radius,
            weights=gaussian_weight_fn,
            algorithm="ball_tree",
            metric="minkowski",
            p=2,
            leaf_size=40,
            n_jobs=-1,
        ),
    }

    for name, model in radius_models.items():
        model.fit(X_train_norm, Y_train)
        Y_pred = predict_radius_regressor_safe(
            model=model,
            X_test_norm=X_test_norm,
            Y_train=Y_train,
            fallback_model=fallback_knn,
            eps=eps,
        )
        metrics = evaluate_predictions(Y_test, Y_pred, mape_eps=mape_eps)

        results[name] = {
            "model": model,
            "Y_pred": Y_pred,
            "metrics": metrics,
        }

    # ========================================================
    # C) Exact custom KDTree Shepard baseline
    # ========================================================

    Y_pred_exact, nn_idx, nn_dist, kd_tree = kd_lookup_predict_exact(
        X_train_norm=X_train_norm,
        Y_train=Y_train,
        X_test_norm=X_test_norm,
        k=k,
        eps=eps,
    )
    metrics_exact = evaluate_predictions(Y_test, Y_pred_exact, mape_eps=mape_eps)

    results["kdtree_shepard_exact"] = {
        "model": kd_tree,
        "Y_pred": Y_pred_exact,
        "metrics": metrics_exact,
        "neighbor_idx": nn_idx,
        "neighbor_dist": nn_dist,
    }

    return results, norm_stats, X_train_norm, X_test_norm


# ============================================================
# 6) Run everything
# ============================================================

k = 5

results, norm_stats, X_train_norm, X_test_norm = run_knn_benchmark(
    X_train=X_train,
    Y_train=Y_train,
    X_test=X_test,
    Y_test=Y_test,
    k=k,
    eps=1e-8,
    mape_eps=1e-8,
)

# ============================================================
# 7) Print full metrics
# ============================================================

for name, obj in results.items():
    print_metrics(name, obj["metrics"])

print_sorted_summary(results)

# ============================================================
# 8) Optional: inspect exact KDTree baseline neighbors
# ============================================================

exact_name = "kdtree_shepard_exact"
if exact_name in results:
    nn_idx = results[exact_name]["neighbor_idx"]
    nn_dist = results[exact_name]["neighbor_dist"]

    print("\nExact KDTree baseline neighbor inspection")
    for i in range(min(3, len(nn_idx))):
        print(f"\nTest sample {i}")
        print("neighbor idx :", nn_idx[i])
        print("neighbor dist:", np.round(nn_dist[i], 6))
        print("prediction   :", np.round(results[exact_name]["Y_pred"][i], 6))

Normalization fitted on training only.
X_train_norm shape: (10000, 24)
X_test_norm shape : (1000, 24)
Using k=5
Estimated Gaussian sigma: 0.132950
Estimated radius         : 0.281351

knn_uniform_l2
Overall MAE : 1.836696 ± 2.740760
Overall RMSE: 3.299276 ± 1.266592
Overall MAPE: 22.841711 ± 38.157982
Target 0: MAE=1.281707 ± 1.079600, RMSE=1.675801 ± 1.079600, MAPE=12.145935 ± 13.051421
Target 1: MAE=1.181260 ± 0.883290, RMSE=1.474983 ± 0.883290, MAPE=19.929207 ± 21.592959
Target 2: MAE=6.484239 ± 4.385874, RMSE=7.828234 ± 4.385874, MAPE=17.296477 ± 13.559917
Target 3: MAE=4.379904 ± 2.839517, RMSE=5.219810 ± 2.839517, MAPE=19.020849 ± 14.685349
Target 4: MAE=1.367400 ± 1.054257, RMSE=1.726627 ± 1.054257, MAPE=9.519287 ± 8.088960
Target 5: MAE=0.349320 ± 0.312848, RMSE=0.468933 ± 0.312848, MAPE=9.478864 ± 7.757337
Target 6: MAE=0.658878 ± 0.443164, RMSE=0.794050 ± 0.443164, MAPE=29.909118 ± 28.297945
Target 7: MAE=0.191573 ± 0.144483, RMSE=0.239949 ± 0.144483, MAPE=29.803719 ± 55.1705

## ANN

In [ ]:
def evaluate_predictions(Y_true, Y_pred, mape_eps=1e-8):
    """
    Return overall and per-target metrics with mean ± std.

    Definitions:
      - Overall MAE mean/std:
          computed from all absolute errors across all samples and targets
      - Overall RMSE mean/std:
          computed from sample-wise RMSE across targets
      - Overall MAPE mean/std:
          computed from all absolute percentage errors across all samples and targets
          using denominator max(|y_true|, mape_eps)
      - Per-target MAE mean/std:
          computed from absolute errors across samples for that target
      - Per-target RMSE mean/std:
          computed from per-sample root squared error for that target
          (for one target this is just absolute error, but we keep the label
           as RMSE for consistency with the original output style)
      - Per-target MAPE mean/std:
          computed from absolute percentage errors across samples for that target
          using denominator max(|y_true|, mape_eps)
    """
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)

    err = Y_pred - Y_true                     # (N, T)
    abs_err = np.abs(err)                    # (N, T)
    sq_err = err ** 2                        # (N, T)

    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0          # (N, T)

    # --------------------------------------------------------
    # Overall MAE = classic global MAE
    # Std = std of absolute errors over all entries
    # --------------------------------------------------------
    overall_mae = abs_err.mean()
    overall_mae_std = abs_err.std(ddof=0)

    # --------------------------------------------------------
    # Overall RMSE = classic global RMSE
    # Std = std of sample-wise RMSE across targets
    # --------------------------------------------------------
    overall_rmse = np.sqrt(sq_err.mean())
    sample_rmse = np.sqrt(sq_err.mean(axis=1))   # (N,)
    overall_rmse_std = sample_rmse.std(ddof=0)

    # --------------------------------------------------------
    # Overall MAPE = classic global MAPE with safe denominator
    # Std = std of absolute percentage errors over all entries
    # --------------------------------------------------------
    overall_mape = ape.mean()
    overall_mape_std = ape.std(ddof=0)

    per_target = []
    for j in range(Y_true.shape[1]):
        abs_j = abs_err[:, j]
        sq_j = sq_err[:, j]
        ape_j = ape[:, j]

        mae_j = abs_j.mean()
        mae_j_std = abs_j.std(ddof=0)

        rmse_j = np.sqrt(sq_j.mean())
        rmse_j_sample = np.sqrt(sq_j)            # = abs error for one target
        rmse_j_std = rmse_j_sample.std(ddof=0)

        mape_j = ape_j.mean()
        mape_j_std = ape_j.std(ddof=0)

        per_target.append({
            "target": j,
            "mae": mae_j,
            "mae_std": mae_j_std,
            "rmse": rmse_j,
            "rmse_std": rmse_j_std,
            "mape": mape_j,
            "mape_std": mape_j_std,
        })

    return {
        "overall_mae": overall_mae,
        "overall_mae_std": overall_mae_std,
        "overall_rmse": overall_rmse,
        "overall_rmse_std": overall_rmse_std,
        "overall_mape": overall_mape,
        "overall_mape_std": overall_mape_std,
        "per_target": per_target,
    }

def print_metrics(name, metrics):
    print(f"\n{name}")
    print(f"Overall MAE : {metrics['overall_mae']:.6f} ± {metrics['overall_mae_std']:.6f}")
    print(f"Overall RMSE: {metrics['overall_rmse']:.6f} ± {metrics['overall_rmse_std']:.6f}")
    print(f"Overall MAPE: {metrics['overall_mape']:.6f} ± {metrics['overall_mape_std']:.6f}")

    for row in metrics["per_target"]:
        print(
            f"Target {row['target']}: "
            f"MAE={row['mae']:.6f} ± {row['mae_std']:.6f}, "
            f"RMSE={row['rmse']:.6f} ± {row['rmse_std']:.6f}, "
            f"MAPE={row['mape']:.6f} ± {row['mape_std']:.6f}"
        )

def build_ann_model(n_features, n_targets, lr=1e-3, dropout=0.15):
    inputs = tf.keras.Input(shape=(n_features,))

    x = tf.keras.layers.Dense(256, activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(dropout)(x)

    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)

    outputs = tf.keras.layers.Dense(n_targets, activation="linear")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="mse",
    )
    return model

In [ ]:
import numpy as np
import tensorflow as tf
from itertools import product
from sklearn.model_selection import train_test_split

# ============================================================
# ANN grid search: 3 levels each for
#   - learning rate
#   - dropout
#   - batch size
#
# Reuses your existing:
#   - fit_typewise_normalizer
#   - transform_typewise
#   - build_ann_model
#   - evaluate_predictions
#   - print_metrics
#
# Uses the same data variables:
#   - X_train, Y_train, X_test, Y_test
# ============================================================


RANDOM_STATE = 42
MAPE_EPS = 1e-8
tf.keras.utils.set_random_seed(RANDOM_STATE)

# ------------------------------------------------------------
# 1) Define 3-level grid
# ------------------------------------------------------------
ann_param_grid = {
    "learning_rate": [0.1, 0.01, 0.15],
    "dropout":       [0.10, 0.20, 0.30],
    "batch_size":    [32, 64, 128],
}

# ------------------------------------------------------------
# 2) Holdout grid search
#    Selection metric = validation overall RMSE
# ------------------------------------------------------------
def ann_grid_search_holdout(
    X_train,
    Y_train,
    val_fraction=0.15,
    param_grid=None,
    max_epochs=300,
    random_state=RANDOM_STATE,
    mape_eps=MAPE_EPS,
):
    if param_grid is None:
        raise ValueError("param_grid must be provided.")

    X_train = np.asarray(X_train, dtype=np.float32)
    Y_train = np.asarray(Y_train, dtype=np.float32)

    if Y_train.ndim == 1:
        Y_train = Y_train.reshape(-1, 1)

    # ============================================================
    # Split RAW data first
    # ============================================================
    X_tr_raw, X_val_raw, Y_tr, Y_val = train_test_split(
        X_train,
        Y_train,
        test_size=val_fraction,
        random_state=random_state,
        shuffle=True,
    )

    # ============================================================
    # Fit normalization using sub-training data ONLY
    # X_tr : trainig set; X_val: validation set;
    # ============================================================
    norm_stats = fit_typewise_normalizer(X_tr_raw)
    X_tr = transform_typewise(
        X_tr_raw,
        norm_stats,
    ).astype(np.float32)

    X_val = transform_typewise(
        X_val_raw,
        norm_stats,
    ).astype(np.float32)


    n_features = X_tr.shape[1]
    n_targets = Y_tr.shape[1]

    rows = []
    best_score = np.inf
    best_params = None
    best_epoch = None

    all_combos = list(product(
        param_grid["learning_rate"],
        param_grid["dropout"],
        param_grid["batch_size"],
    ))

    print(f"Total ANN combinations: {len(all_combos)}")

    for run_id, (lr, dropout, batch_size) in enumerate(all_combos, start=1):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(random_state)

        model = build_ann_model(
            n_features=n_features,
            n_targets=n_targets,
            lr=lr,
            dropout=dropout,
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=20,
                restore_best_weights=True,
                verbose=0,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=6,
                min_lr=1e-6,
                verbose=0,
            ),
        ]

        history = model.fit(
            X_tr,
            Y_tr,
            validation_data=(X_val, Y_val),
            epochs=max_epochs,
            batch_size=batch_size,
            verbose=0,
            callbacks=callbacks,
        )

        # best epoch based on val_loss
        best_epoch_this_run = int(np.argmin(history.history["val_loss"])) + 1

        # predict validation
        Y_val_pred = model.predict(X_val, batch_size=batch_size, verbose=0)

        # back to original target scale
        Y_val_pred = np.asarray(Y_val_pred, dtype=np.float64)

        val_metrics = evaluate_predictions(
            Y_true=Y_val,
            Y_pred=Y_val_pred,
            mape_eps=mape_eps,
        )

        val_rmse = val_metrics["overall_rmse"]
        val_mae  = val_metrics["overall_mae"]
        val_mape = val_metrics["overall_mape"]

        row = {
            "run_id": run_id,
            "learning_rate": lr,
            "dropout": dropout,
            "batch_size": batch_size,
            "best_epoch": best_epoch_this_run,
            "val_rmse": val_rmse,
            "val_mae": val_mae,
            "val_mape": val_mape,
        }
        rows.append(row)

        print(
            f"[{run_id:02d}/{len(all_combos)}] "
            f"lr={lr:.0e}, dropout={dropout:.2f}, batch={batch_size:3d} | "
            f"val RMSE={val_rmse:.6f}, MAE={val_mae:.6f}, "
            f"MAPE={val_mape:.6f}, best_epoch={best_epoch_this_run}"
        )

        # select best by validation RMSE
        if val_rmse < best_score:
            best_score = val_rmse
            best_params = {
                "learning_rate": lr,
                "dropout": dropout,
                "batch_size": batch_size,
            }
            best_epoch = best_epoch_this_run

    # sort summary by best validation RMSE
    rows = sorted(rows, key=lambda x: x["val_rmse"])

    return {
        "rows": rows,
        "best_score": best_score,
        "best_params": best_params,
        "best_epoch": best_epoch,    }


# ------------------------------------------------------------
# 3) Refit best ANN on full training set, then test
# ------------------------------------------------------------
def ann_refit_best_on_full_train(
    X_train,
    Y_train,
    X_test,
    Y_test,
    best_params,
    best_epoch,
    random_state=RANDOM_STATE,
    mape_eps=MAPE_EPS,
):
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test  = np.asarray(X_test, dtype=np.float32)
    Y_train = np.asarray(Y_train, dtype=np.float32)
    Y_test  = np.asarray(Y_test, dtype=np.float32)

    if Y_train.ndim == 1:
        Y_train = Y_train.reshape(-1, 1)
    if Y_test.ndim == 1:
        Y_test = Y_test.reshape(-1, 1)

    # training-only X normalization on full train
    norm_stats = fit_typewise_normalizer(X_train)
    X_train_norm = transform_typewise(X_train, norm_stats).astype(np.float32)
    X_test_norm  = transform_typewise(X_test, norm_stats).astype(np.float32)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(random_state)

    model = build_ann_model(
        n_features=X_train_norm.shape[1],
        n_targets=Y_train.shape[1],
        lr=best_params["learning_rate"],
        dropout=best_params["dropout"],
    )

    history = model.fit(
        X_train_norm,
        Y_train,
        epochs=best_epoch,
        batch_size=best_params["batch_size"],
        verbose=1,
    )

    Y_pred_test = model.predict(
        X_test_norm,
        batch_size=best_params["batch_size"],
        verbose=0,
    )
    Y_pred_test = np.asarray(Y_pred_test, dtype=np.float64)

    test_metrics = evaluate_predictions(
        Y_true=Y_test,
        Y_pred=Y_pred_test,
        mape_eps=mape_eps,
    )

    return {
        "model": model,
        "history": history.history,
        "Y_pred": Y_pred_test,
        "metrics": test_metrics,
        "norm_stats": norm_stats,
        "best_params": best_params,
        "best_epoch": best_epoch,
    }


# ------------------------------------------------------------
# 4) Run ANN grid search
# ------------------------------------------------------------
ann_search_out = ann_grid_search_holdout(
    X_train=X_train,
    Y_train=Y_train,
    val_fraction=0.15,
    param_grid=ann_param_grid,
    max_epochs=300,
    random_state=RANDOM_STATE,
    mape_eps=MAPE_EPS,
)

print("\nBest ANN config")
print("best_params   :", ann_search_out["best_params"])
print("best_val_rmse :", f"{ann_search_out['best_score']:.6f}")
print("best_epoch    :", ann_search_out["best_epoch"])

# optional: print top 10 configs
print("\nTop 10 ANN configs by validation RMSE")
for row in ann_search_out["rows"][:10]:
    print(
        f"lr={row['learning_rate']:.0e}, "
        f"dropout={row['dropout']:.2f}, "
        f"batch={row['batch_size']:3d}, "
        f"best_epoch={row['best_epoch']:3d}, "
        f"val_RMSE={row['val_rmse']:.6f}, "
        f"val_MAE={row['val_mae']:.6f}, "
        f"val_MAPE={row['val_mape']:.6f}"
    )


# ------------------------------------------------------------
# 5) Refit best ANN on full training set and evaluate on test
# ------------------------------------------------------------
ann_best_full = ann_refit_best_on_full_train(
    X_train=X_train,
    Y_train=Y_train,
    X_test=X_test,
    Y_test=Y_test,
    best_params=ann_search_out["best_params"],
    best_epoch=ann_search_out["best_epoch"],
    random_state=RANDOM_STATE,
    mape_eps=MAPE_EPS,
)

if "results" not in globals():
    results = {}

results["ann_keras_tuned"] = ann_best_full
results["ann_keras_tuned"]["search_rows"] = ann_search_out["rows"]
results["ann_keras_tuned"]["best_val_rmse"] = ann_search_out["best_score"]

print_metrics("ann_keras_tuned", ann_best_full["metrics"])
print("best_params   :", ann_best_full["best_params"])
print("best_epoch    :", ann_best_full["best_epoch"])
print("best_val_rmse :", f"{ann_search_out['best_score']:.6f}")

Total ANN combinations: 27
[01/27] lr=1e-01, dropout=0.10, batch= 32 | val RMSE=3.602434, MAE=2.047976, MAPE=28.781212, best_epoch=31
[02/27] lr=1e-01, dropout=0.10, batch= 64 | val RMSE=3.485500, MAE=1.833275, MAPE=22.904924, best_epoch=45
[03/27] lr=1e-01, dropout=0.10, batch=128 | val RMSE=3.458400, MAE=1.760907, MAPE=21.365815, best_epoch=82
[04/27] lr=1e-01, dropout=0.20, batch= 32 | val RMSE=3.472070, MAE=1.803478, MAPE=22.377557, best_epoch=115
[05/27] lr=1e-01, dropout=0.20, batch= 64 | val RMSE=3.798913, MAE=2.328030, MAPE=33.646024, best_epoch=10
[06/27] lr=1e-01, dropout=0.20, batch=128 | val RMSE=3.468104, MAE=1.800565, MAPE=22.890729, best_epoch=65
[07/27] lr=1e-01, dropout=0.30, batch= 32 | val RMSE=3.651292, MAE=2.148809, MAPE=35.356697, best_epoch=18
[08/27] lr=1e-01, dropout=0.30, batch= 64 | val RMSE=3.461207, MAE=1.776380, MAPE=21.740776, best_epoch=132
[09/27] lr=1e-01, dropout=0.30, batch=128 | val RMSE=3.459911, MAE=1.790339, MAPE=22.289293, best_epoch=131
[10/27]

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# ============================================================
# Fixed ANN training with:
#   learning_rate = 0.01
#   dropout       = 0.10
#   batch_size    = 128
# Validation split = 15% of training set
#
# Reuses your existing:
#   - fit_typewise_normalizer
#   - transform_typewise
#   - evaluate_predictions
#   - print_metrics
# and your data:
#   - X_train, Y_train, X_test, Y_test
# ============================================================

RANDOM_STATE = 42
MAPE_EPS = 1e-8
ANN_PARAMS = {
    "learning_rate": 0.15,
    "dropout": 0.10,
    "batch_size": 64,
}

tf.keras.utils.set_random_seed(RANDOM_STATE)


def run_ann_fixed_15pct_val(
    X_train,
    Y_train,
    X_test,
    Y_test,
    learning_rate=0.01,
    dropout=0.10,
    batch_size=64,
    val_fraction=0.15,
    max_epochs=300,
    random_state=RANDOM_STATE,
    mape_eps=MAPE_EPS,
):
    # --------------------------------------------------------
    # safety / shapes
    # --------------------------------------------------------
    X_train = np.asarray(X_train, dtype=np.float32)
    X_test  = np.asarray(X_test, dtype=np.float32)
    Y_train = np.asarray(Y_train, dtype=np.float32)
    Y_test  = np.asarray(Y_test, dtype=np.float32)

    if Y_train.ndim == 1:
        Y_train = Y_train.reshape(-1, 1)
    if Y_test.ndim == 1:
        Y_test = Y_test.reshape(-1, 1)


    # --------------------------------------------------------
    # split training into train/validation
    # --------------------------------------------------------
    X_tr, X_val, Y_tr, Y_val = train_test_split(
        np.asarray(X_train, dtype=np.float64),
        np.asarray(Y_train, dtype=np.float64),
        test_size=0.15,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    # --------------------------------------------------------
    # X normalization: fit on full training set only
    # --------------------------------------------------------
    norm_stats = fit_typewise_normalizer(X_tr)
    X_tr = transform_typewise(X_tr, norm_stats).astype(np.float32)
    X_val = transform_typewise(X_val, norm_stats).astype(np.float32)
    X_test_norm  = transform_typewise(X_test, norm_stats).astype(np.float32)

    # --------------------------------------------------------
    # build and train model
    # --------------------------------------------------------
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(random_state)

    model = build_ann_model(
        n_features=X_tr.shape[1],
        n_targets=Y_tr.shape[1],
        lr=learning_rate,
        dropout=dropout,
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_tr,
        Y_tr,
        validation_data=(X_val, Y_val),
        epochs=max_epochs,
        batch_size=batch_size,
        verbose=1,
        callbacks=callbacks,
    )

    best_epoch = int(np.argmin(history.history["val_loss"])) + 1

    # --------------------------------------------------------
    # validation prediction
    # --------------------------------------------------------
    Y_val_pred = model.predict(X_val, batch_size=batch_size, verbose=0)
    Y_val_pred = np.asarray(Y_val_pred, dtype=np.float64)

    val_metrics = evaluate_predictions(
        Y_true=Y_val,
        Y_pred=Y_val_pred,
        mape_eps=mape_eps,
    )

    # --------------------------------------------------------
    # test prediction
    # --------------------------------------------------------
    Y_test_pred = model.predict(
        X_test_norm,
        batch_size=batch_size,
        verbose=0,
    )

    Y_test_pred = np.asarray(Y_test_pred, dtype=np.float64)

    test_metrics = evaluate_predictions(
        Y_true=Y_test,
        Y_pred=Y_test_pred,
        mape_eps=mape_eps,
    )

    return {
        "model": model,
        "history": history.history,
        "norm_stats": norm_stats,
        "Y_val_pred": Y_val_pred,
        "Y_test_pred": Y_test_pred,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "best_epoch": best_epoch,
        "params": {
            "learning_rate": learning_rate,
            "dropout": dropout,
            "batch_size": batch_size,
            "val_fraction": val_fraction,
        },
    }


# ============================================================
# Run fixed ANN
# ============================================================

ann_fixed = run_ann_fixed_15pct_val(
    X_train=X_train,
    Y_train=Y_train,
    X_test=X_test,
    Y_test=Y_test,
    learning_rate=ANN_PARAMS["learning_rate"],
    dropout=ANN_PARAMS["dropout"],
    batch_size=ANN_PARAMS["batch_size"],
    val_fraction=0.15,
    max_epochs=300,
    random_state=RANDOM_STATE,
    mape_eps=MAPE_EPS,
)

print("\nANN fixed params")
print("params     :", ann_fixed["params"])
print("best_epoch :", ann_fixed["best_epoch"])

print_metrics("ann_fixed_validation_15pct", ann_fixed["val_metrics"])
print_metrics("ann_fixed_test", ann_fixed["test_metrics"])

# optional: store in your results dict
if "results" not in globals():
    results = {}

results["ann_fixed_15pct_val"] = {
    "model": ann_fixed["model"],
    "Y_pred": ann_fixed["Y_test_pred"],
    "val_metrics": ann_fixed["val_metrics"],
    "metrics": ann_fixed["test_metrics"],
    "best_epoch": ann_fixed["best_epoch"],
    "params": ann_fixed["params"],
}

Epoch 1/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - loss: 27.1771 - val_loss: 24.6018 - learning_rate: 0.1500
Epoch 2/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.7089 - val_loss: 19.8870 - learning_rate: 0.1500
Epoch 3/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.1997 - val_loss: 37.5742 - learning_rate: 0.1500
Epoch 4/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.0889 - val_loss: 22.4024 - learning_rate: 0.1500
Epoch 5/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.5048 - val_loss: 17.5227 - learning_rate: 0.1500
Epoch 6/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 15.3111 - val_loss: 26.4702 - learning_rate: 0.1500
Epoch 7/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.8004 - val_loss: 28.3240 - learning_rate: 0.1500
Epoch 8/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.4086 - val_loss: 29.6994 - learning_rate: 0.1500
Epoch 9/300
133/133 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 14.0463 - val_loss: 22.0694 - lea

### Boost methods

In [ ]:
# ============================================================
# REPLACEMENT BLOCK: GPU-only boosting models
# Trains ONLY:
#   - XGBoost (gbtree, dart)
#   - LightGBM (gbdt, dart)
#   - CatBoost
#
# Reuses:
#   - results
#   - X_train_norm, X_test_norm
#   - Y_train, Y_test
#   - evaluate_predictions
#   - print_metrics
#   - print_sorted_summary
# ============================================================
! pip install lightgbm --config-settings=cmake.define.USE_CUDA=ON


import numpy as np
from sklearn.multioutput import MultiOutputRegressor

# -------------------------
# Optional third-party imports
# -------------------------
HAS_XGB = True
HAS_LGBM = True
HAS_CAT = True

try:
    from xgboost import XGBRegressor
except Exception as e:
    HAS_XGB = False
    print(f"[skip] xgboost not available: {e}")

try:
    from lightgbm import LGBMRegressor
except Exception as e:
    HAS_LGBM = False
    print(f"[skip] lightgbm not available: {e}")

try:
    from catboost import CatBoostRegressor
except Exception as e:
    HAS_CAT = False
    print(f"[skip] catboost not available: {e}")

RANDOM_STATE = 42

# -------------------------
# Helpers
# -------------------------
def _wrap_multioutput_if_needed(base_estimator, Y):
    """
    Wrap estimator for multi-output regression when needed.
    XGBoost / LightGBM need this for multi-target regression.
    """
    Y = np.asarray(Y)
    if Y.ndim == 1 or (Y.ndim == 2 and Y.shape[1] == 1):
        return base_estimator
    return MultiOutputRegressor(base_estimator, n_jobs=-1)


def _fit_predict_store(name, model, X_train_norm, Y_train, X_test_norm, Y_test, results):
    """
    Fit model, predict, evaluate, and append into results dict.
    """
    model.fit(X_train_norm, Y_train)
    Y_pred = model.predict(X_test_norm)

    Y_pred = np.asarray(Y_pred)
    if Y_pred.ndim == 1 and np.asarray(Y_test).ndim == 2 and Y_test.shape[1] == 1:
        Y_pred = Y_pred.reshape(-1, 1)

    metrics = evaluate_predictions(Y_test, Y_pred)

    results[name] = {
        "model": model,
        "Y_pred": Y_pred,
        "metrics": metrics,
    }
    print(f"[done] {name}")


[skip] catboost not available: No module named 'catboost'


In [ ]:
# ============================================================
# EXTRA CODE: holdout validation grid search
# Models used:
#   - XGBoost      (GPU)
#   - LightGBM     (GPU)
#   - CatBoost     (GPU)
#   - RandomForest (CPU)
# ============================================================

# If LightGBM GPU is not yet built in your environment, install it first.
# !pip install -U lightgbm --config-settings=cmake.define.USE_CUDA=ON

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
REPORT_NAME = "radius_distance_l2"   # change this label if you want
MAPE_EPS = 1e-8

# ------------------------------------------------------------
# 0) Metric helpers with MAPE
# ------------------------------------------------------------
def _ensure_2d(a):
    a = np.asarray(a)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    return a

def _overall_rmse(Y_true, Y_pred):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((Y_true - Y_pred) ** 2)))

def _overall_mape(Y_true, Y_pred, eps=1e-8):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)
    denom = np.maximum(np.abs(Y_true), eps)
    return float(np.mean(np.abs(Y_pred - Y_true) / denom) * 100.0)

def evaluate_predictions(Y_true, Y_pred, mape_eps=1e-8):
    """
    Return overall and per-target metrics with mean ± std.

    Definitions:
      - Overall MAE mean/std:
          computed from all absolute errors across all samples and targets
      - Overall RMSE mean/std:
          computed from sample-wise RMSE across targets
      - Overall MAPE mean/std:
          computed from all absolute percentage errors across all samples and targets
          using denominator max(|y_true|, mape_eps)
      - Per-target MAE mean/std:
          computed from absolute errors across samples for that target
      - Per-target RMSE mean/std:
          computed from per-sample root squared error for that target
          (for one target this is just absolute error, but we keep the label
           as RMSE for consistency with the original output style)
      - Per-target MAPE mean/std:
          computed from absolute percentage errors across samples for that target
          using denominator max(|y_true|, mape_eps)
    """
    Y_true = _ensure_2d(np.asarray(Y_true, dtype=np.float64))
    Y_pred = _ensure_2d(np.asarray(Y_pred, dtype=np.float64))

    err = Y_pred - Y_true
    abs_err = np.abs(err)
    sq_err = err ** 2

    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0

    overall_mae = float(abs_err.mean())
    overall_mae_std = float(abs_err.std(ddof=0))

    overall_rmse = float(np.sqrt(sq_err.mean()))
    sample_rmse = np.sqrt(sq_err.mean(axis=1))
    overall_rmse_std = float(sample_rmse.std(ddof=0))

    overall_mape = float(ape.mean())
    overall_mape_std = float(ape.std(ddof=0))

    per_target = []
    for j in range(Y_true.shape[1]):
        abs_j = abs_err[:, j]
        sq_j = sq_err[:, j]
        ape_j = ape[:, j]

        mae_j = float(abs_j.mean())
        mae_j_std = float(abs_j.std(ddof=0))

        rmse_j = float(np.sqrt(sq_j.mean()))
        rmse_j_std = float(np.sqrt(sq_j).std(ddof=0))

        mape_j = float(ape_j.mean())
        mape_j_std = float(ape_j.std(ddof=0))

        per_target.append({
            "target": j,
            "mae": mae_j,
            "mae_std": mae_j_std,
            "rmse": rmse_j,
            "rmse_std": rmse_j_std,
            "mape": mape_j,
            "mape_std": mape_j_std,
        })

    return {
        "overall_mae": overall_mae,
        "overall_mae_std": overall_mae_std,
        "overall_rmse": overall_rmse,
        "overall_rmse_std": overall_rmse_std,
        "overall_mape": overall_mape,
        "overall_mape_std": overall_mape_std,
        "per_target": per_target,
    }

def print_metrics(name, metrics):
    print(f"\n{name}")
    print(f"Overall MAE : {metrics['overall_mae']:.6f} ± {metrics['overall_mae_std']:.6f}")
    print(f"Overall RMSE: {metrics['overall_rmse']:.6f} ± {metrics['overall_rmse_std']:.6f}")
    print(f"Overall MAPE: {metrics['overall_mape']:.6f} ± {metrics['overall_mape_std']:.6f}")

    for row in metrics["per_target"]:
        print(
            f"Target {row['target']}: "
            f"MAE={row['mae']:.6f} ± {row['mae_std']:.6f}, "
            f"RMSE={row['rmse']:.6f} ± {row['rmse_std']:.6f}, "
            f"MAPE={row['mape']:.6f} ± {row['mape_std']:.6f}"
        )

def print_sorted_summary(results):
    """
    Print a compact summary sorted by RMSE.
    """
    print("\n" + "=" * 130)
    print("Sorted summary (best overall RMSE first)")
    print("=" * 130)

    rows = []
    for name, obj in results.items():
        m = obj["metrics"]
        rows.append((
            name,
            m["overall_mae"],
            m["overall_mae_std"],
            m["overall_rmse"],
            m["overall_rmse_std"],
            m["overall_mape"],
            m["overall_mape_std"],
        ))

    rows.sort(key=lambda x: x[3])

    for name, mae_, mae_std_, rmse_, rmse_std_, mape_, mape_std_ in rows:
        print(
            f"{name:30s}  "
            f"MAE={mae_:10.6f} ± {mae_std_:10.6f}  "
            f"RMSE={rmse_:10.6f} ± {rmse_std_:10.6f}  "
            f"MAPE={mape_:10.6f} ± {mape_std_:10.6f}"
        )

def _detailed_error_summary(Y_true, Y_pred, mape_eps=1e-8):
    """
    Returns a dict with:
      - overall MAE mean ± std(|error|)
      - overall RMSE ± std(per-sample RMSE)
      - overall MAPE ± std(absolute percentage error)
      - per-target MAE/RMSE/MAPE ± std
    """
    Y_true = _ensure_2d(np.asarray(Y_true, dtype=np.float64))
    Y_pred = _ensure_2d(np.asarray(Y_pred, dtype=np.float64))

    err = Y_pred - Y_true
    abs_err = np.abs(err)
    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0

    overall_mae = float(abs_err.mean())
    overall_mae_std = float(abs_err.std())

    overall_rmse = float(np.sqrt(np.mean(err ** 2)))
    per_sample_rmse = np.sqrt(np.mean(err ** 2, axis=1))
    overall_rmse_std = float(per_sample_rmse.std())

    overall_mape = float(ape.mean())
    overall_mape_std = float(ape.std())

    by_target = []
    for j in range(Y_true.shape[1]):
        tgt_abs = abs_err[:, j]
        tgt_err = err[:, j]
        tgt_ape = ape[:, j]

        tgt_mae = float(tgt_abs.mean())
        tgt_mae_std = float(tgt_abs.std())

        tgt_rmse = float(np.sqrt(np.mean(tgt_err ** 2)))
        tgt_rmse_std = tgt_mae_std

        tgt_mape = float(tgt_ape.mean())
        tgt_mape_std = float(tgt_ape.std())

        by_target.append({
            "target": j,
            "mae": tgt_mae,
            "mae_std": tgt_mae_std,
            "rmse": tgt_rmse,
            "rmse_std": tgt_rmse_std,
            "mape": tgt_mape,
            "mape_std": tgt_mape_std,
        })

    return {
        "overall_mae": overall_mae,
        "overall_mae_std": overall_mae_std,
        "overall_rmse": overall_rmse,
        "overall_rmse_std": overall_rmse_std,
        "overall_mape": overall_mape,
        "overall_mape_std": overall_mape_std,
        "by_target": by_target,
    }

def print_detailed_test_report(name, Y_true, Y_pred, mape_eps=1e-8):
    s = _detailed_error_summary(Y_true, Y_pred, mape_eps=mape_eps)

    print(name)
    print(f"Overall MAE : {s['overall_mae']:.6f} ± {s['overall_mae_std']:.6f}")
    print(f"Overall RMSE: {s['overall_rmse']:.6f} ± {s['overall_rmse_std']:.6f}")
    print(f"Overall MAPE: {s['overall_mape']:.6f} ± {s['overall_mape_std']:.6f}")

    for row in s["by_target"]:
        j = row["target"]
        print(
            f"Target {j}: "
            f"MAE={row['mae']:.6f} ± {row['mae_std']:.6f}, "
            f"RMSE={row['rmse']:.6f} ± {row['rmse_std']:.6f}, "
            f"MAPE={row['mape']:.6f} ± {row['mape_std']:.6f}"
        )

# ------------------------------------------------------------
# 1) Holdout validation split from TRAIN only
# ------------------------------------------------------------
X_tr, X_val, Y_tr, Y_val = train_test_split(
    np.asarray(X_train, dtype=np.float64),
    np.asarray(Y_train, dtype=np.float64),
    test_size=0.15,
    random_state=RANDOM_STATE,
    shuffle=True,
)

print("Holdout validation split")
print("X_tr shape :", X_tr.shape)
print("X_val shape:", X_val.shape)
print("Y_tr shape :", Y_tr.shape)
print("Y_val shape:", Y_val.shape)

# ------------------------------------------------------------
# 2) Model builder
# ------------------------------------------------------------
def _build_tunable_model(model_name, params, Y_for_shape):
    """
    GPU for boosting models, CPU for RandomForest.
    """
    if model_name == "xgboost":
        base = XGBRegressor(
            objective="reg:squarederror",
            booster="gbtree",
            tree_method="hist",
            device="cuda",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
            learning_rate=params.get("learning_rate", 0.05),
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", 6),
        )
        return _wrap_multioutput_if_needed(base, Y_for_shape)

    elif model_name == "lightgbm":
        base = LGBMRegressor(
            boosting_type="gbdt",
            objective="regression",
            device_type="cuda",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
            learning_rate=params.get("learning_rate", 0.05),
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", -1),
        )
        return _wrap_multioutput_if_needed(base, Y_for_shape)

    elif model_name == "catboost":
        Y_arr = np.asarray(Y_for_shape)
        multi_target = (Y_arr.ndim == 2 and Y_arr.shape[1] > 1)

        common_kwargs = dict(
            random_seed=RANDOM_STATE,
            verbose=False,
            task_type="GPU",
            devices="0",
            learning_rate=params.get("learning_rate", 0.05),
            iterations=params.get("iterations", 500),
            depth=params.get("depth", 6),
        )

        if multi_target:
            return CatBoostRegressor(
                loss_function="MultiRMSE",
                eval_metric="MultiRMSE",
                **common_kwargs,
            )
        else:
            return CatBoostRegressor(
                loss_function="RMSE",
                eval_metric="RMSE",
                **common_kwargs,
            )

    elif model_name == "random_forest":
        return RandomForestRegressor(
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", None),
            min_samples_split=params.get("min_samples_split", 2),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    else:
        raise ValueError(
            f"{model_name} is unsupported. "
            "Use only: xgboost, lightgbm, catboost, random_forest."
        )

# ------------------------------------------------------------
# 3) Grid search helper
# ------------------------------------------------------------
def _grid_search_holdout(model_name, param_grid, X_tr, Y_tr, X_val, Y_val, mape_eps=1e-8):
    """
    Manual grid search using validation RMSE.
    If a config fails, it is skipped.
    """
    all_rows = []
    best_score = np.inf
    best_params = None
    best_model = None

    combos = list(ParameterGrid(param_grid))
    print(f"\n[{model_name}] trying {len(combos)} combinations")

    for i, params in enumerate(combos, start=1):
        try:
            model = _build_tunable_model(model_name, params, Y_tr)
            model.fit(X_tr, Y_tr)
            Y_val_pred = model.predict(X_val)

            val_rmse = _overall_rmse(Y_val, Y_val_pred)
            val_mae = mean_absolute_error(Y_val, Y_val_pred)
            val_mape = _overall_mape(Y_val, Y_val_pred, eps=mape_eps)

            row = {
                "params": params,
                "val_rmse": float(val_rmse),
                "val_mae": float(val_mae),
                "val_mape": float(val_mape),
                "status": "ok",
            }
            all_rows.append(row)

            print(
                f"[{model_name}] {i:02d}/{len(combos):02d} | "
                f"val_rmse={val_rmse:.6f} | val_mae={val_mae:.6f} | val_mape={val_mape:.6f} | params={params}"
            )

            if val_rmse < best_score:
                best_score = val_rmse
                best_params = params
                best_model = model

        except Exception as e:
            row = {
                "params": params,
                "val_rmse": np.inf,
                "val_mae": np.inf,
                "val_mape": np.inf,
                "status": f"failed: {str(e)}",
            }
            all_rows.append(row)

            print(
                f"[{model_name}] {i:02d}/{len(combos):02d} | "
                f"FAILED | params={params} | error={e}"
            )

    if best_params is None:
        raise RuntimeError(
            f"No valid configuration succeeded for {model_name}."
        )

    return {
        "best_score": best_score,
        "best_params": best_params,
        "best_model_on_split": best_model,
        "rows": all_rows,
    }

def _refit_best_on_full_train(model_name, best_params, X_train_full, Y_train_full):
    model = _build_tunable_model(model_name, best_params, Y_train_full)
    model.fit(X_train_full, Y_train_full)
    return model

Holdout validation split
X_tr shape : (8500, 24)
X_val shape: (1500, 24)
Y_tr shape : (8500, 9)
Y_val shape: (1500, 9)


In [ ]:
# ============================================================
# EXTRA CODE: holdout validation grid search
# Models used:
#   - XGBoost      (GPU)
#   - LightGBM     (GPU)
#   - CatBoost     (GPU)
#   - RandomForest (CPU)
# ============================================================

# If LightGBM GPU is not yet built in your environment, install it first.
# !pip install -U lightgbm --config-settings=cmake.define.USE_CUDA=ON

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
REPORT_NAME = "radius_distance_l2"   # change this label if you want
MAPE_EPS = 1e-8

# ------------------------------------------------------------
# 0) Metric helpers with MAPE
# ------------------------------------------------------------
def _ensure_2d(a):
    a = np.asarray(a)
    if a.ndim == 1:
        a = a.reshape(-1, 1)
    return a

def _overall_rmse(Y_true, Y_pred):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)
    return float(np.sqrt(np.mean((Y_true - Y_pred) ** 2)))

def _overall_mape(Y_true, Y_pred, eps=1e-8):
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)
    denom = np.maximum(np.abs(Y_true), eps)
    return float(np.mean(np.abs(Y_pred - Y_true) / denom) * 100.0)

def evaluate_predictions(Y_true, Y_pred, mape_eps=1e-8):
    """
    Return overall and per-target metrics with mean ± std.

    Definitions:
      - Overall MAE mean/std:
          computed from all absolute errors across all samples and targets
      - Overall RMSE mean/std:
          computed from sample-wise RMSE across targets
      - Overall MAPE mean/std:
          computed from all absolute percentage errors across all samples and targets
          using denominator max(|y_true|, mape_eps)
      - Per-target MAE mean/std:
          computed from absolute errors across samples for that target
      - Per-target RMSE mean/std:
          computed from per-sample root squared error for that target
          (for one target this is just absolute error, but we keep the label
           as RMSE for consistency with the original output style)
      - Per-target MAPE mean/std:
          computed from absolute percentage errors across samples for that target
          using denominator max(|y_true|, mape_eps)
    """
    Y_true = _ensure_2d(np.asarray(Y_true, dtype=np.float64))
    Y_pred = _ensure_2d(np.asarray(Y_pred, dtype=np.float64))

    err = Y_pred - Y_true
    abs_err = np.abs(err)
    sq_err = err ** 2

    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0

    overall_mae = float(abs_err.mean())
    overall_mae_std = float(abs_err.std(ddof=0))

    overall_rmse = float(np.sqrt(sq_err.mean()))
    sample_rmse = np.sqrt(sq_err.mean(axis=1))
    overall_rmse_std = float(sample_rmse.std(ddof=0))

    overall_mape = float(ape.mean())
    overall_mape_std = float(ape.std(ddof=0))

    per_target = []
    for j in range(Y_true.shape[1]):
        abs_j = abs_err[:, j]
        sq_j = sq_err[:, j]
        ape_j = ape[:, j]

        mae_j = float(abs_j.mean())
        mae_j_std = float(abs_j.std(ddof=0))

        rmse_j = float(np.sqrt(sq_j.mean()))
        rmse_j_std = float(np.sqrt(sq_j).std(ddof=0))

        mape_j = float(ape_j.mean())
        mape_j_std = float(ape_j.std(ddof=0))

        per_target.append({
            "target": j,
            "mae": mae_j,
            "mae_std": mae_j_std,
            "rmse": rmse_j,
            "rmse_std": rmse_j_std,
            "mape": mape_j,
            "mape_std": mape_j_std,
        })

    return {
        "overall_mae": overall_mae,
        "overall_mae_std": overall_mae_std,
        "overall_rmse": overall_rmse,
        "overall_rmse_std": overall_rmse_std,
        "overall_mape": overall_mape,
        "overall_mape_std": overall_mape_std,
        "per_target": per_target,
    }

def print_metrics(name, metrics):
    print(f"\n{name}")
    print(f"Overall MAE : {metrics['overall_mae']:.6f} ± {metrics['overall_mae_std']:.6f}")
    print(f"Overall RMSE: {metrics['overall_rmse']:.6f} ± {metrics['overall_rmse_std']:.6f}")
    print(f"Overall MAPE: {metrics['overall_mape']:.6f} ± {metrics['overall_mape_std']:.6f}")

    for row in metrics["per_target"]:
        print(
            f"Target {row['target']}: "
            f"MAE={row['mae']:.6f} ± {row['mae_std']:.6f}, "
            f"RMSE={row['rmse']:.6f} ± {row['rmse_std']:.6f}, "
            f"MAPE={row['mape']:.6f} ± {row['mape_std']:.6f}"
        )

def print_sorted_summary(results):
    """
    Print a compact summary sorted by RMSE.
    """
    print("\n" + "=" * 130)
    print("Sorted summary (best overall RMSE first)")
    print("=" * 130)

    rows = []
    for name, obj in results.items():
        m = obj["metrics"]
        rows.append((
            name,
            m["overall_mae"],
            m["overall_mae_std"],
            m["overall_rmse"],
            m["overall_rmse_std"],
            m["overall_mape"],
            m["overall_mape_std"],
        ))

    rows.sort(key=lambda x: x[3])

    for name, mae_, mae_std_, rmse_, rmse_std_, mape_, mape_std_ in rows:
        print(
            f"{name:30s}  "
            f"MAE={mae_:10.6f} ± {mae_std_:10.6f}  "
            f"RMSE={rmse_:10.6f} ± {rmse_std_:10.6f}  "
            f"MAPE={mape_:10.6f} ± {mape_std_:10.6f}"
        )

def _detailed_error_summary(Y_true, Y_pred, mape_eps=1e-8):
    """
    Returns a dict with:
      - overall MAE mean ± std(|error|)
      - overall RMSE ± std(per-sample RMSE)
      - overall MAPE ± std(absolute percentage error)
      - per-target MAE/RMSE/MAPE ± std
    """
    Y_true = _ensure_2d(np.asarray(Y_true, dtype=np.float64))
    Y_pred = _ensure_2d(np.asarray(Y_pred, dtype=np.float64))

    err = Y_pred - Y_true
    abs_err = np.abs(err)
    denom = np.maximum(np.abs(Y_true), mape_eps)
    ape = (abs_err / denom) * 100.0

    overall_mae = float(abs_err.mean())
    overall_mae_std = float(abs_err.std())

    overall_rmse = float(np.sqrt(np.mean(err ** 2)))
    per_sample_rmse = np.sqrt(np.mean(err ** 2, axis=1))
    overall_rmse_std = float(per_sample_rmse.std())

    overall_mape = float(ape.mean())
    overall_mape_std = float(ape.std())

    by_target = []
    for j in range(Y_true.shape[1]):
        tgt_abs = abs_err[:, j]
        tgt_err = err[:, j]
        tgt_ape = ape[:, j]

        tgt_mae = float(tgt_abs.mean())
        tgt_mae_std = float(tgt_abs.std())

        tgt_rmse = float(np.sqrt(np.mean(tgt_err ** 2)))
        tgt_rmse_std = tgt_mae_std

        tgt_mape = float(tgt_ape.mean())
        tgt_mape_std = float(tgt_ape.std())

        by_target.append({
            "target": j,
            "mae": tgt_mae,
            "mae_std": tgt_mae_std,
            "rmse": tgt_rmse,
            "rmse_std": tgt_rmse_std,
            "mape": tgt_mape,
            "mape_std": tgt_mape_std,
        })

    return {
        "overall_mae": overall_mae,
        "overall_mae_std": overall_mae_std,
        "overall_rmse": overall_rmse,
        "overall_rmse_std": overall_rmse_std,
        "overall_mape": overall_mape,
        "overall_mape_std": overall_mape_std,
        "by_target": by_target,
    }

def print_detailed_test_report(name, Y_true, Y_pred, mape_eps=1e-8):
    s = _detailed_error_summary(Y_true, Y_pred, mape_eps=mape_eps)

    print(name)
    print(f"Overall MAE : {s['overall_mae']:.6f} ± {s['overall_mae_std']:.6f}")
    print(f"Overall RMSE: {s['overall_rmse']:.6f} ± {s['overall_rmse_std']:.6f}")
    print(f"Overall MAPE: {s['overall_mape']:.6f} ± {s['overall_mape_std']:.6f}")

    for row in s["by_target"]:
        j = row["target"]
        print(
            f"Target {j}: "
            f"MAE={row['mae']:.6f} ± {row['mae_std']:.6f}, "
            f"RMSE={row['rmse']:.6f} ± {row['rmse_std']:.6f}, "
            f"MAPE={row['mape']:.6f} ± {row['mape_std']:.6f}"
        )

# ------------------------------------------------------------
# 1) Holdout validation split from TRAIN only
# ------------------------------------------------------------
X_tr, X_val, Y_tr, Y_val = train_test_split(
    np.asarray(X_train, dtype=np.float64),
    np.asarray(Y_train, dtype=np.float64),
    test_size=0.15,
    random_state=RANDOM_STATE,
    shuffle=True,
)

print("Holdout validation split")
print("X_tr shape :", X_tr.shape)
print("X_val shape:", X_val.shape)
print("Y_tr shape :", Y_tr.shape)
print("Y_val shape:", Y_val.shape)

# ------------------------------------------------------------
# 2) Model builder
# ------------------------------------------------------------
def _build_tunable_model(model_name, params, Y_for_shape):
    """
    GPU for boosting models, CPU for RandomForest.
    """
    if model_name == "xgboost":
        base = XGBRegressor(
            objective="reg:squarederror",
            booster="gbtree",
            tree_method="hist",
            device="cuda",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0,
            learning_rate=params.get("learning_rate", 0.05),
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", 6),
        )
        return _wrap_multioutput_if_needed(base, Y_for_shape)

    elif model_name == "lightgbm":
        base = LGBMRegressor(
            boosting_type="gbdt",
            objective="regression",
            device_type="cuda",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
            learning_rate=params.get("learning_rate", 0.05),
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", -1),
        )
        return _wrap_multioutput_if_needed(base, Y_for_shape)

    elif model_name == "catboost":
        Y_arr = np.asarray(Y_for_shape)
        multi_target = (Y_arr.ndim == 2 and Y_arr.shape[1] > 1)

        common_kwargs = dict(
            random_seed=RANDOM_STATE,
            verbose=False,
            task_type="GPU",
            devices="0",
            learning_rate=params.get("learning_rate", 0.05),
            iterations=params.get("iterations", 500),
            depth=params.get("depth", 6),
        )

        if multi_target:
            return CatBoostRegressor(
                loss_function="MultiRMSE",
                eval_metric="MultiRMSE",
                **common_kwargs,
            )
        else:
            return CatBoostRegressor(
                loss_function="RMSE",
                eval_metric="RMSE",
                **common_kwargs,
            )

    elif model_name == "random_forest":
        return RandomForestRegressor(
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", None),
            min_samples_split=params.get("min_samples_split", 2),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    else:
        raise ValueError(
            f"{model_name} is unsupported. "
            "Use only: xgboost, lightgbm, catboost, random_forest."
        )

# ------------------------------------------------------------
# 3) Grid search helper
# ------------------------------------------------------------
def _grid_search_holdout(model_name, param_grid, X_tr, Y_tr, X_val, Y_val, mape_eps=1e-8):
    """
    Manual grid search using validation RMSE.
    If a config fails, it is skipped.
    """
    all_rows = []
    best_score = np.inf
    best_params = None
    best_model = None

    combos = list(ParameterGrid(param_grid))
    print(f"\n[{model_name}] trying {len(combos)} combinations")

    for i, params in enumerate(combos, start=1):
        try:
            model = _build_tunable_model(model_name, params, Y_tr)
            model.fit(X_tr, Y_tr)
            Y_val_pred = model.predict(X_val)

            val_rmse = _overall_rmse(Y_val, Y_val_pred)
            val_mae = mean_absolute_error(Y_val, Y_val_pred)
            val_mape = _overall_mape(Y_val, Y_val_pred, eps=mape_eps)

            row = {
                "params": params,
                "val_rmse": float(val_rmse),
                "val_mae": float(val_mae),
                "val_mape": float(val_mape),
                "status": "ok",
            }
            all_rows.append(row)

            print(
                f"[{model_name}] {i:02d}/{len(combos):02d} | "
                f"val_rmse={val_rmse:.6f} | val_mae={val_mae:.6f} | val_mape={val_mape:.6f} | params={params}"
            )

            if val_rmse < best_score:
                best_score = val_rmse
                best_params = params
                best_model = model

        except Exception as e:
            row = {
                "params": params,
                "val_rmse": np.inf,
                "val_mae": np.inf,
                "val_mape": np.inf,
                "status": f"failed: {str(e)}",
            }
            all_rows.append(row)

            print(
                f"[{model_name}] {i:02d}/{len(combos):02d} | "
                f"FAILED | params={params} | error={e}"
            )

    if best_params is None:
        raise RuntimeError(
            f"No valid configuration succeeded for {model_name}."
        )

    return {
        "best_score": best_score,
        "best_params": best_params,
        "best_model_on_split": best_model,
        "rows": all_rows,
    }

def _refit_best_on_full_train(model_name, best_params, X_train_full, Y_train_full):
    model = _build_tunable_model(model_name, best_params, Y_train_full)
    model.fit(X_train_full, Y_train_full)
    return model

# ------------------------------------------------------------
# 4) Parameter grids
# ------------------------------------------------------------
param_grids = {}

if HAS_XGB:
    param_grids["xgboost"] = {
        "learning_rate": [0.03, 0.05, 0.1],
        "n_estimators": [200, 500, 1000],
        "max_depth": [4, 6, 8],
    }

if HAS_LGBM:
    param_grids["lightgbm"] = {
        "learning_rate": [0.03, 0.05, 0.1],
        "n_estimators": [200, 500, 1000],
        "max_depth": [-1, 6, 10],
    }

if HAS_CAT:
    param_grids["catboost"] = {
        "learning_rate": [0.03, 0.05, 0.1],
        "iterations": [200, 500, 1000],
        "depth": [4, 6, 8],
    }

param_grids["random_forest"] = {
    "n_estimators": [200, 500, 1000],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
}

# ------------------------------------------------------------
# 5) Choose models
# ------------------------------------------------------------
models_to_tune = []

if HAS_XGB:
    models_to_tune.append("xgboost")
if HAS_LGBM:
    models_to_tune.append("lightgbm")
if HAS_CAT:
    models_to_tune.append("catboost")

models_to_tune.append("random_forest")

print("\nModels to tune:", models_to_tune)

# ------------------------------------------------------------
# 6) Run holdout grid search
# ------------------------------------------------------------
tuning_results = {}

for model_name in models_to_tune:
    try:
        search_out = _grid_search_holdout(
            model_name=model_name,
            param_grid=param_grids[model_name],
            X_tr=X_tr,
            Y_tr=Y_tr,
            X_val=X_val,
            Y_val=Y_val,
            mape_eps=MAPE_EPS,
        )
        tuning_results[model_name] = search_out

        print(f"\nBest for {model_name}")
        print(f"  best val RMSE : {search_out['best_score']:.6f}")
        print(f"  best params   : {search_out['best_params']}")

    except Exception as e:
        print(f"\n[skip] {model_name} could not be tuned: {e}")

# ------------------------------------------------------------
# 7) Refit best models on FULL training set, then test once
# ------------------------------------------------------------
for model_name, search_out in tuning_results.items():
    best_params = search_out["best_params"]

    best_full_model = _refit_best_on_full_train(
    model_name=model_name,
    best_params=best_params,
    X_train_full=np.asarray(X_train, dtype=np.float64),
    Y_train_full=np.asarray(Y_train, dtype=np.float64),
    )

    Y_pred_test = best_full_model.predict(
        np.asarray(X_test, dtype=np.float64)
    )
    metrics = evaluate_predictions(Y_test, Y_pred_test, mape_eps=MAPE_EPS)

    tuned_name = f"{model_name}_tuned"

    results[tuned_name] = {
        "model": best_full_model,
        "Y_pred": Y_pred_test,
        "metrics": metrics,
        "best_params": best_params,
        "best_val_rmse": search_out["best_score"],
        "search_rows": search_out["rows"],
    }

    print(f"\n[done] {tuned_name}")
    print(f"best params   : {best_params}")
    print(f"best val RMSE : {search_out['best_score']:.6f}")

# ------------------------------------------------------------
# 8) Print tuned-model results only
# ------------------------------------------------------------
print("\n" + "=" * 96)
print("Tuned model results")
print("=" * 96)

for model_name in models_to_tune:
    tuned_name = f"{model_name}_tuned"
    if tuned_name in results:
        print_metrics(tuned_name, results[tuned_name]["metrics"])
        print("best_params   :", results[tuned_name]["best_params"])
        print("best_val_rmse :", f"{results[tuned_name]['best_val_rmse']:.6f}")

# ------------------------------------------------------------
# 9) Reprint full combined summary
# ------------------------------------------------------------
print_sorted_summary(results)

# ------------------------------------------------------------
# 10) Print detailed test-set report for the SINGLE BEST combo
# ------------------------------------------------------------
successful_tuned = [
    name for name in results.keys()
    if name.endswith("_tuned") and "best_val_rmse" in results[name]
]

if len(successful_tuned) == 0:
    raise RuntimeError("No tuned model succeeded, so there is no best combination to report.")

best_tuned_name = min(
    successful_tuned,
    key=lambda name: results[name]["best_val_rmse"]
)

best_entry = results[best_tuned_name]
best_pred = best_entry["Y_pred"]

print("\n" + "=" * 96)
print("Best combination at the very end")
print("=" * 96)
print("Best tuned model :", best_tuned_name)
print("Best params      :", best_entry["best_params"])
print("Best val RMSE    :", f"{best_entry['best_val_rmse']:.6f}")
print()

print_detailed_test_report(REPORT_NAME, Y_test, best_pred, mape_eps=MAPE_EPS)

Holdout validation split
X_tr shape : (8500, 24)
X_val shape: (1500, 24)
Y_tr shape : (8500, 9)
Y_val shape: (1500, 9)

Models to tune: ['xgboost', 'lightgbm', 'random_forest']

[xgboost] trying 27 combinations
[xgboost] 01/27 | val_rmse=3.629101 | val_mae=2.154830 | val_mape=30.044304 | params={'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 200}
[xgboost] 02/27 | val_rmse=3.546628 | val_mae=1.983419 | val_mape=25.825568 | params={'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 500}
[xgboost] 03/27 | val_rmse=3.510433 | val_mae=1.872125 | val_mape=22.739843 | params={'learning_rate': 0.03, 'max_depth': 4, 'n_estimators': 1000}
[xgboost] 04/27 | val_rmse=3.567177 | val_mae=2.018214 | val_mape=26.752997 | params={'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 200}
[xgboost] 05/27 | val_rmse=3.533438 | val_mae=1.888883 | val_mape=23.178279 | params={'learning_rate': 0.03, 'max_depth': 6, 'n_estimators': 500}
[xgboost] 06/27 | val_rmse=3.555570 | val_mae=1.838512 | v

### CatBoost

In [ ]:
# ============================================================
# EXTRA CODE: rerun ONLY CatBoost on CPU
# Assumes your previous code has already defined:
#   - _build_tunable_model
#   - _grid_search_holdout
#   - _refit_best_on_full_train
#   - evaluate_predictions
#   - print_metrics
#   - X_tr, X_val, Y_tr, Y_val
#   - X_train_norm, X_test_norm
#   - Y_train, Y_test
#   - param_grids
#   - results
#   - tuning_results
#   - RANDOM_STATE, MAPE_EPS
#   - HAS_CAT
# ============================================================
! pip install catboost
# ============================================================
# EXTRA CODE: fix CatBoost availability, add its grid,
# and run CatBoost on CPU with grid search
# ============================================================

from catboost import CatBoostRegressor

# mark CatBoost as available now
HAS_CAT = True

# create CatBoost grid if it does not exist yet
if "param_grids" not in globals():
    param_grids = {}

param_grids["catboost"] = {
    "learning_rate": [0.03, 0.05, 0.1],
    "iterations": [200, 500, 1000],
    "depth": [4, 6, 8],
}

# keep old builder
_old_build_tunable_model = _build_tunable_model

# override only CatBoost -> CPU
def _build_tunable_model(model_name, params, Y_for_shape):
    if model_name == "catboost":
        Y_arr = np.asarray(Y_for_shape)
        multi_target = (Y_arr.ndim == 2 and Y_arr.shape[1] > 1)

        common_kwargs = dict(
            random_seed=RANDOM_STATE,
            verbose=False,
            task_type="CPU",
            thread_count=-1,
            learning_rate=params.get("learning_rate", 0.05),
            iterations=params.get("iterations", 500),
            depth=params.get("depth", 6),
        )

        if multi_target:
            return CatBoostRegressor(
                loss_function="MultiRMSE",
                eval_metric="MultiRMSE",
                **common_kwargs,
            )
        else:
            return CatBoostRegressor(
                loss_function="RMSE",
                eval_metric="RMSE",
                **common_kwargs,
            )

    return _old_build_tunable_model(model_name, params, Y_for_shape)

# run CatBoost-only grid search on CPU
cat_search_out = _grid_search_holdout(
    model_name="catboost",
    param_grid=param_grids["catboost"],
    X_tr=X_tr,
    Y_tr=Y_tr,
    X_val=X_val,
    Y_val=Y_val,
    mape_eps=MAPE_EPS,
)

print("\nBest for catboost_cpu")
print(f"  best val RMSE : {cat_search_out['best_score']:.6f}")
print(f"  best params   : {cat_search_out['best_params']}")

# Get the final results:
cat_best_full_model = _refit_best_on_full_train(
    model_name="catboost",
    best_params=cat_search_out["best_params"],
    X_train_full=X_train,
    Y_train_full=Y_train,
)

Y_pred_test_cat_cpu = cat_best_full_model.predict(X_test)
cat_cpu_metrics = evaluate_predictions(
    Y_test,
    Y_pred_test_cat_cpu,
    mape_eps=MAPE_EPS,
)

results["catboost_cpu_tuned"] = {
    "model": cat_best_full_model,
    "Y_pred": Y_pred_test_cat_cpu,
    "metrics": cat_cpu_metrics,
    "best_params": cat_search_out["best_params"],
    "best_val_rmse": cat_search_out["best_score"],
    "search_rows": cat_search_out["rows"],
}

if "tuning_results" not in globals():
    tuning_results = {}

tuning_results["catboost_cpu_tuned"] = cat_search_out

print_metrics("catboost_cpu_tuned", results["catboost_cpu_tuned"]["metrics"])
print("best_params   :", results["catboost_cpu_tuned"]["best_params"])
print("best_val_rmse :", f"{results['catboost_cpu_tuned']['best_val_rmse']:.6f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 25.0 MB/s eta 0:00:00

[catboost] trying 27 combinations
[catboost] 01/27 | val_rmse=3.797328 | val_mae=2.408507 | val_mape=35.192936 | params={'depth': 4, 'iterations': 200, 'learning_rate': 0.03}
[catboost] 02/27 | val_rmse=3.751934 | val_mae=2.346163 | val_mape=34.007195 | params={'depth': 4, 'iterations': 200, 'learning_rate': 0.05}
[catboost] 03/27 | val_rmse=3.678702 | val_mae=2.243991 | val_mape=32.084413 | params={'depth': 4, 'iterations': 200, 'learning_rate': 0.1}
[catboost] 04/27 | val_rmse=3.713728 | val_mae=2.293989 | val_mape=33.098095 | params={'depth': 4, 'iterations': 500, 'learning_rate': 0.03}
[catboost] 05/27 | val_rmse=3.646224 | val_mae=2.200502 | val_mape=31.229470 | params={'depth': 4, 'iterations': 500, 'learning_rate': 0.05}
[catboost] 06/27 | val_rmse=3.568785 | val_mae=2.064429 | val_mape=28.128281 | params={'depth': 4, 'iterations': 500, 'learning_rate': 0.1}
[catboost] 07/27 | val_rmse=3.630668 | val

In [ ]:
# ============================================================
# EXTRA CODE: add and tune sklearn Gradient Boosting (CPU)
# Assumes these already exist from your previous code:
#   - _build_tunable_model
#   - _grid_search_holdout
#   - _refit_best_on_full_train
#   - evaluate_predictions
#   - print_metrics
#   - X_tr, X_val, Y_tr, Y_val
#   - X_train_norm, X_test_norm
#   - Y_train, Y_test
#   - param_grids
#   - results
#   - tuning_results
#   - RANDOM_STATE, MAPE_EPS
#   - _wrap_multioutput_if_needed
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor

# create grid if it does not exist yet
if "param_grids" not in globals():
    param_grids = {}

param_grids["gradient_boost"] = {
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [200, 500, 1000],
    "max_depth": [2, 3, 4],
}

# keep current builder
_old_build_tunable_model_gb = _build_tunable_model

# override only gradient_boost
def _build_tunable_model(model_name, params, Y_for_shape):
    if model_name == "gradient_boost":
        base = GradientBoostingRegressor(
            learning_rate=params.get("learning_rate", 0.05),
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", 3),
            random_state=RANDOM_STATE,
        )
        return _wrap_multioutput_if_needed(base, Y_for_shape)

    return _old_build_tunable_model_gb(model_name, params, Y_for_shape)

# run Gradient Boosting-only grid search
gb_search_out = _grid_search_holdout(
    model_name="gradient_boost",
    param_grid=param_grids["gradient_boost"],
    X_tr=X_tr,
    Y_tr=Y_tr,
    X_val=X_val,
    Y_val=Y_val,
    mape_eps=MAPE_EPS,
)

print("\nBest for gradient_boost")
print(f"  best val RMSE : {gb_search_out['best_score']:.6f}")
print(f"  best params   : {gb_search_out['best_params']}")

# refit on full training set
gb_best_full_model = _refit_best_on_full_train(
    model_name="gradient_boost",
    best_params=gb_search_out["best_params"],
    X_train_full=X_train,
    Y_train_full=Y_train,
)

Y_pred_test_gb = gb_best_full_model.predict(X_test)
gb_metrics = evaluate_predictions(
    Y_test,
    Y_pred_test_gb,
    mape_eps=MAPE_EPS,
)

results["gradient_boost_tuned"] = {
    "model": gb_best_full_model,
    "Y_pred": Y_pred_test_gb,
    "metrics": gb_metrics,
    "best_params": gb_search_out["best_params"],
    "best_val_rmse": gb_search_out["best_score"],
    "search_rows": gb_search_out["rows"],
}

if "tuning_results" not in globals():
    tuning_results = {}

tuning_results["gradient_boost_tuned"] = gb_search_out

print_metrics("gradient_boost_tuned", results["gradient_boost_tuned"]["metrics"])
print("best_params   :", results["gradient_boost_tuned"]["best_params"])
print("best_val_rmse :", f"{results['gradient_boost_tuned']['best_val_rmse']:.6f}")


[gradient_boost] trying 27 combinations
[gradient_boost] 01/27 | val_rmse=3.741648 | val_mae=2.339834 | val_mape=33.938234 | params={'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 200}
[gradient_boost] 02/27 | val_rmse=3.645483 | val_mae=2.189629 | val_mape=30.687857 | params={'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 500}
[gradient_boost] 03/27 | val_rmse=3.573949 | val_mae=2.057012 | val_mape=27.482020 | params={'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 1000}
[gradient_boost] 04/27 | val_rmse=3.688535 | val_mae=2.257115 | val_mape=32.133815 | params={'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 200}
[gradient_boost] 05/27 | val_rmse=3.587498 | val_mae=2.085191 | val_mape=28.137852 | params={'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 500}
[gradient_boost] 06/27 | val_rmse=3.536628 | val_mae=1.957737 | val_mape=24.895055 | params={'learning_rate': 0.03, 'max_depth': 3, 'n_estimators': 1000}
[gradient_boost] 07/27 | val_rmse=3.627